In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
import os
import gc

# Project paths
PROJECT_PATH = r"C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026"
TRAIN_PATH = os.path.join(PROJECT_PATH, "data", "raw", "train")
TEST_PATH = os.path.join(PROJECT_PATH, "data", "raw", "test")

print("Project:", PROJECT_PATH)
print("Training data:", TRAIN_PATH)
print("Imports successful.")

Project: C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026
Training data: C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\data\raw\train
Imports successful.


In [3]:
# Load Source 1 only

s1_path = os.path.join(TRAIN_PATH, "train_source1.tsv")

train1 = pd.read_csv(
    s1_path,
    sep="\t"
)

print("S1 shape:", train1.shape)
print("\nColumns:")
print(train1.columns.tolist())

print("\nMemory usage:")
print(f"{train1.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

S1 shape: (2206821, 4)

Columns:
['entity_id', 'business_name', 'business_address', 'country']

Memory usage:
604.43 MB


In [4]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM:      {memory.total / 1024**3:.2f} GB")
print(f"Available RAM:  {memory.available / 1024**3:.2f} GB")
print(f"Used RAM:       {memory.percent}%")

Total RAM:      15.63 GB
Available RAM:  6.31 GB
Used RAM:       59.6%


In [5]:
gt_path = os.path.join(TRAIN_PATH, "train_ground_truth.tsv")

train_gr = pd.read_csv(
    gt_path,
    sep="\t"
)

print("GT shape:", train_gr.shape)
print("\nColumns:")
print(train_gr.columns.tolist())

print("\nMemory usage:")
print(f"{train_gr.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

GT shape: (2206821, 2)

Columns:
['source1_entity_id', 'matched_entity_ids']

Memory usage:
321.17 MB


In [6]:
# Basic GT statistics

print("Missing matched_entity_ids:",
      train_gr["matched_entity_ids"].isna().sum())

print("\nRows with matches:",
      train_gr["matched_entity_ids"].notna().sum())

print("\nSample GT:")
display(train_gr.head(10))

Missing matched_entity_ids: 123247

Rows with matches: 2083574

Sample GT:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [7]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKC", text)

    text = "".join(
        " " if unicodedata.category(ch).startswith(("P", "S")) else ch
        for ch in text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


# Normalize S1
train1["name_norm"] = train1["business_name"].map(normalize_text)
train1["address_norm"] = train1["business_address"].map(normalize_text)

# Compact name
train1["name_compact"] = (
    train1["name_norm"]
    .str.replace(" ", "", regex=False)
)

# Blocking key 1:
# country + first 4 characters of compact name
train1["name_prefix"] = train1["name_compact"].str[:4]

train1["block_key"] = (
    train1["country"].fillna("").astype(str)
    + "_"
    + train1["name_prefix"]
)

# Blocking key 2:
# first 8 characters of compact address
train1["address_compact"] = (
    train1["address_norm"]
    .str.replace(" ", "", regex=False)
)

train1["address_prefix"] = (
    train1["address_compact"].str[:8]
)

print("S1 blocking columns created.")
print(train1[
    [
        "entity_id",
        "business_name",
        "name_norm",
        "address_prefix",
        "block_key"
    ]
].head())

S1 blocking columns created.
      entity_id            business_name                name_norm  \
0  S1-925783039      Orelee's Barbershop      orelee s barbershop   
1  S1-773889195              Prime Money              prime money   
2  S1-377745466            B+ Retail Inc             b retail inc   
3  S1-133037285            Christ Chapel            christ chapel   
4  S1-755362802  Prabhav Business Center  prabhav business center   

  address_prefix   block_key  
0       1795west     US_orel  
1       17560ell     US_prim  
2       1712mont     US_bret  
3       2100came     US_chri  
4       797laket  India_prab  


In [8]:
# Extract the first two numeric groups from each address.
# This returns strings, not Python lists.

address_numbers = train1["business_address"].fillna("").astype(str).str.findall(r"\d+")

train1["address_numbers"] = (
    address_numbers.str[:2].str.join("_")
)

train1["first_address_number"] = (
    address_numbers.str[0].fillna("")
)

# Country + first address number
train1["number_block"] = (
    train1["country"].fillna("").astype(str)
    + "_"
    + train1["first_address_number"]
)

print("Address-number blocking columns created.")

print(
    train1[
        [
            "business_address",
            "address_numbers",
            "first_address_number",
            "number_block"
        ]
    ].head(10)
)

Address-number blocking columns created.
                                    business_address address_numbers  \
0             1795 Westchester Drive, High Point, NC            1795   
1                    17560 Ellis Road, Tahlequah, OK           17560   
2                1712 Montebello Avenue, Phoenix, AZ            1712   
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD            2100   
4  797, Lake Town Block A, Kolkata, Howrah, West ...             797   
5                  OH, Columbus, 5559 Orville Avenue            5559   
6  2505, Tower 1, Oakwood, Runwal Greens, Mulund ...          2505_1   
7       1111 Church Street, Unit 2007, Nashville, TN       1111_2007   
8              337 Oakland Avenue, Michigan City, IN             337   
9  294 Meadowcreek Drive, Unit Unit 2, Village Of...           294_2   

  first_address_number number_block  
0                 1795      US_1795  
1                17560     US_17560  
2                 1712      US_1712  
3             

In [11]:
train1.head(2)

,entity_id,business_name,business_address,country,name_norm,address_norm,name_compact,name_prefix,block_key,address_compact,address_prefix,address_numbers,first_address_number,number_block
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester drive high point nc,oreleesbarbershop,orel,US_orel,1795westchesterdrivehighpointnc,1795west,1795,1795,US_1795
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis road tahlequah ok,primemoney,prim,US_prim,17560ellisroadtahlequahok,17560ell,17560,17560,US_17560


In [12]:
del address_numbers
gc.collect()

print("Temporary memory released.")

Temporary memory released.


In [13]:
memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / 1024**3:.2f} GB")
print(f"RAM usage:     {memory.percent:.1f}%")

Available RAM: 4.51 GB
RAM usage:     71.2%


In [14]:
s2_path = os.path.join(TRAIN_PATH, "train_source2.tsv")

s2_test = pd.read_csv(
    s2_path,
    sep="\t",
    nrows=10_000
)

print("S2 test shape:", s2_test.shape)
print("\nColumns:")
print(s2_test.columns.tolist())

print("\nMemory:")
print(f"{s2_test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

display(s2_test.head())

S2 test shape: (10000, 4)

Columns:
['entity_id', 'business_name', 'business_address', 'country']

Memory:
2.81 MB


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [16]:
S2_CHUNK_SIZE = 50_000

s2_chunk = pd.read_csv(
    s2_path,
    sep="\t",
    nrows=S2_CHUNK_SIZE
)

print("Chunk shape:", s2_chunk.shape)

print(
    f"Chunk memory: "
    f"{s2_chunk.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

# Create only the blocking columns we need
s2_chunk["name_norm"] = (
    s2_chunk["business_name"]
    .fillna("")
    .astype(str)
    .str.lower()
)

s2_chunk["name_compact"] = (
    s2_chunk["name_norm"]
    .str.replace(r"[\W_]+", "", regex=True)
)

s2_chunk["name_prefix"] = (
    s2_chunk["name_compact"].str[:4]
)

s2_chunk["block_key"] = (
    s2_chunk["country"].fillna("").astype(str)
    + "_"
    + s2_chunk["name_prefix"]
)

s2_chunk["address_compact"] = (
    s2_chunk["business_address"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"[\W_]+", "", regex=True)
)

s2_chunk["address_prefix"] = (
    s2_chunk["address_compact"].str[:8]
)

print("\nBlocking columns created successfully.")
print(
    s2_chunk[
        [
            "entity_id",
            "business_name",
            "block_key",
            "address_prefix"
        ]
    ].head()
)

print(
    f"\nFinal chunk memory: "
    f"{s2_chunk.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Chunk shape: (50000, 4)
Chunk memory: 14.07 MB

Blocking columns created successfully.
      entity_id                    business_name   block_key address_prefix
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड  India_रममर       khno5701
1  S2-764573417     -- Holloway Peak Inc Seafood     US_holl       105elmst
2  S2-639257739         आदित्य प्रॉपर्टीज एलएलपी  India_आदतय       g3571gul
3  S2-163963287                       Summit Inc     US_summ       greensbo
4   S2-49942811     Delta Tetlecommunication Inc     US_delt       914pierp

Final chunk memory: 33.90 MB


In [17]:
del s2_chunk
gc.collect()

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / 1024**3:.2f} GB")
print(f"RAM usage:     {memory.percent:.1f}%")

Available RAM: 3.37 GB
RAM usage:     78.5%


In [18]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKC", text)

    text = "".join(
        " " if unicodedata.category(ch).startswith(("P", "S")) else ch
        for ch in text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


print(normalize_text("Orelee's Barbershop"))
print(normalize_text("B+ Retail Inc"))
print(normalize_text("SHIVSHAKTI VIDYALAYA | www.shivshakti.com"))

orelee s barbershop
b retail inc
shivshakti vidyalaya www shivshakti com


In [19]:
import shutil

total, used, free = shutil.disk_usage(PROJECT_PATH)

print(f"Total disk: {total / 1024**3:.2f} GB")
print(f"Used disk:  {used / 1024**3:.2f} GB")
print(f"Free disk:  {free / 1024**3:.2f} GB")

Total disk: 181.16 GB
Used disk:  120.06 GB
Free disk:  61.10 GB


In [20]:
# Remove temporary S2 test data if still present
for var in ["s2_test", "s2_chunk"]:
    if var in globals():
        del globals()[var]

gc.collect()

print("Temporary objects cleaned.")

Temporary objects cleaned.


In [21]:
import psutil

process = psutil.Process()

ram = psutil.virtual_memory()

print(f"System RAM available: {ram.available / 1024**3:.2f} GB")
print(f"System RAM used:      {ram.percent:.1f}%")

print(
    f"\ntrain1 memory: "
    f"{train1.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

System RAM available: 2.85 GB
System RAM used:      81.8%

train1 memory: 1.94 GB


In [22]:
# Columns that are no longer needed in RAM
columns_to_drop = [
    "name_norm",
    "address_norm",
    "name_compact",
    "name_prefix",
    "address_compact"
]

# Drop only if they exist
columns_to_drop = [
    col for col in columns_to_drop
    if col in train1.columns
]

train1.drop(columns=columns_to_drop, inplace=True)

gc.collect()

print("Dropped:", columns_to_drop)
print("\nRemaining columns:")
print(train1.columns.tolist())

print(
    f"\ntrain1 memory: "
    f"{train1.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

Dropped: ['name_norm', 'address_norm', 'name_compact', 'name_prefix', 'address_compact']

Remaining columns:
['entity_id', 'business_name', 'business_address', 'country', 'block_key', 'address_prefix', 'address_numbers', 'first_address_number', 'number_block']

train1 memory: 1.16 GB


In [23]:
BLOCKING_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "blocking"
)

os.makedirs(BLOCKING_PATH, exist_ok=True)

print("Blocking directory:")
print(BLOCKING_PATH)

print("\nContents:")
print(os.listdir(BLOCKING_PATH))

Blocking directory:
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\data\processed\blocking

Contents:
[]


In [24]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq

    print("PyArrow is available.")
    print("PyArrow version:", pa.__version__)

except ImportError:
    print("PyArrow is NOT installed.")

PyArrow is NOT installed.


In [25]:
import sqlite3

DB_PATH = os.path.join(
    BLOCKING_PATH,
    "entity_blocking.db"
)

conn = sqlite3.connect(DB_PATH)

print("SQLite database created:")
print(DB_PATH)

SQLite database created:
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\data\processed\blocking\entity_blocking.db


In [26]:
conn.execute("""
CREATE TABLE IF NOT EXISTS entity_blocks (
    entity_id TEXT,
    source TEXT,
    block_key TEXT,
    address_prefix TEXT,
    address_numbers TEXT,
    number_block TEXT
)
""")

conn.commit()

print("SQLite table created successfully.")

SQLite table created successfully.


In [27]:
tables = conn.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
""").fetchall()

print(tables)

[('entity_blocks',)]


In [30]:
def extract_address_numbers(text):
    if pd.isna(text):
        return ""

    numbers = re.findall(r'\d+', str(text))
    return "_".join(numbers[:2])


def extract_first_address_number(text):
    if pd.isna(text):
        return ""

    numbers = re.findall(r'\d+', str(text))

    if len(numbers) == 0:
        return ""

    return numbers[0]

print("Address helper functions restored.")

Address helper functions restored.


In [31]:
s2_test["address_numbers"] = (
    s2_test["business_address"]
    .map(extract_address_numbers)
)

s2_test["first_address_number"] = (
    s2_test["business_address"]
    .map(extract_first_address_number)
)

s2_test["number_block"] = (
    s2_test["country"].fillna("").astype(str)
    + "_"
    + s2_test["first_address_number"].fillna("")
)

print("Blocking features completed.")

print(
    s2_test[
        [
            "entity_id",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ].head()
)

Blocking features completed.
      entity_id   block_key address_prefix address_numbers number_block
0  S2-166376419  India_रामम       khno5701          570_13    India_570
1  S2-764573417     US_holl       105elmst             105       US_105
2  S2-639257739  India_आदित       g3571gul           3_571      India_3
3  S2-163963287     US_summ       greensbo            19_1        US_19
4   S2-49942811     US_delt       914pierp             914       US_914


In [32]:
# Empty first-number blocks must not be used for candidate generation
s2_test.loc[
    s2_test["first_address_number"].eq(""),
    "number_block"
] = ""

print(
    "Rows with usable number_block:",
    (s2_test["number_block"] != "").sum()
)

print(
    "Rows with empty number_block:",
    (s2_test["number_block"] == "").sum()
)

Rows with usable number_block: 917
Rows with empty number_block: 83


In [33]:
s2_db = s2_test[
    [
        "entity_id",
        "block_key",
        "address_prefix",
        "address_numbers",
        "number_block"
    ]
].copy()

s2_db["source"] = "S2"

s2_db = s2_db[
    [
        "entity_id",
        "source",
        "block_key",
        "address_prefix",
        "address_numbers",
        "number_block"
    ]
]

s2_db.to_sql(
    "entity_blocks",
    conn,
    if_exists="append",
    index=False,
    chunksize=1000
)

conn.commit()

print("1,000 S2 rows inserted into SQLite.")

1,000 S2 rows inserted into SQLite.


In [34]:
count = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
WHERE source = 'S2'
""").fetchone()[0]

print("S2 rows in database:", count)

S2 rows in database: 1000


In [35]:
print("Creating indexes...")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_block_key
ON entity_blocks(block_key)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_address_prefix
ON entity_blocks(address_prefix)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_address_numbers
ON entity_blocks(address_numbers)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_number_block
ON entity_blocks(number_block)
""")

conn.commit()

print("All 4 indexes created successfully.")

Creating indexes...
All 4 indexes created successfully.


In [36]:
indexes = conn.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'index'
AND tbl_name = 'entity_blocks'
""").fetchall()

print("Indexes:")
for index in indexes:
    print(index[0])

Indexes:
idx_block_key
idx_address_prefix
idx_address_numbers
idx_number_block


In [38]:
# Take the first S1 entity
s1_row = train1.iloc[0]

print("S1 entity:", s1_row["entity_id"])
print("Name:", s1_row["business_name"])
print("Country:", s1_row["country"])

print("\nBlocking keys:")
print("block_key:", s1_row["block_key"])
print("address_prefix:", s1_row["address_prefix"])
print("address_numbers:", s1_row["address_numbers"])
print("number_block:", s1_row["number_block"])

S1 entity: S1-925783039
Name: Orelee's Barbershop
Country: US

Blocking keys:
block_key: US_orel
address_prefix: 1795west
address_numbers: 1795
number_block: US_1795


In [39]:
# Look up candidates using the name block
name_candidates = conn.execute(
    """
    SELECT entity_id, source
    FROM entity_blocks
    WHERE block_key = ?
    LIMIT 20
    """,
    (s1_row["block_key"],)
).fetchall()

print("Name-block candidates found:", len(name_candidates))

for candidate in name_candidates[:10]:
    print(candidate)

Name-block candidates found: 1
('S2-762023797', 'S2')


In [40]:
conn.execute("""
DELETE FROM entity_blocks
WHERE source = 'S2'
""")

conn.commit()

count = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
""").fetchone()[0]

print("Rows remaining in database:", count)

Rows remaining in database: 0


In [41]:
import time
import gc

S2_CHUNK_SIZE = 50_000

s2_start_time = time.time()
s2_rows_processed = 0
s2_chunk_number = 0

print("Starting full S2 processing...")
print(f"Chunk size: {S2_CHUNK_SIZE:,}")
print()

for chunk in pd.read_csv(
    s2_path,
    sep="\t",
    chunksize=S2_CHUNK_SIZE
):
    s2_chunk_number += 1
    chunk_start = time.time()

    # --------------------------------------------------
    # 1. Normalize name and address
    # --------------------------------------------------
    chunk["name_norm"] = chunk["business_name"].map(normalize_text)
    chunk["address_norm"] = chunk["business_address"].map(normalize_text)

    # --------------------------------------------------
    # 2. Name blocking key
    # country + first 4 compact name characters
    # --------------------------------------------------
    chunk["name_compact"] = (
        chunk["name_norm"]
        .str.replace(" ", "", regex=False)
    )

    chunk["name_prefix"] = (
        chunk["name_compact"].str[:4]
    )

    chunk["block_key"] = (
        chunk["country"].fillna("").astype(str)
        + "_"
        + chunk["name_prefix"].fillna("")
    )

    # --------------------------------------------------
    # 3. Address prefix blocking key
    # first 8 compact address characters
    # --------------------------------------------------
    chunk["address_compact"] = (
        chunk["address_norm"]
        .str.replace(" ", "", regex=False)
    )

    chunk["address_prefix"] = (
        chunk["address_compact"].str[:8]
    )

    # --------------------------------------------------
    # 4. Address number blocking key
    # first two address numbers
    # --------------------------------------------------
    chunk["address_numbers"] = (
        chunk["business_address"]
        .map(extract_address_numbers)
    )

    # --------------------------------------------------
    # 5. First address number blocking key
    # country + first address number
    # --------------------------------------------------
    chunk["first_address_number"] = (
        chunk["business_address"]
        .map(extract_first_address_number)
    )

    chunk["number_block"] = (
        chunk["country"].fillna("").astype(str)
        + "_"
        + chunk["first_address_number"].fillna("")
    )

    # IMPORTANT:
    # Empty number blocks such as US_ / India_
    # must never be used as blocking keys.
    chunk.loc[
        chunk["first_address_number"].eq(""),
        "number_block"
    ] = ""

    # --------------------------------------------------
    # 6. Keep ONLY compact database columns
    # --------------------------------------------------
    db_chunk = chunk[
        [
            "entity_id",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ].copy()

    db_chunk["source"] = "S2"

    db_chunk = db_chunk[
        [
            "entity_id",
            "source",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ]

    # --------------------------------------------------
    # 7. Write to SQLite
    # --------------------------------------------------
    db_chunk.to_sql(
        "entity_blocks",
        conn,
        if_exists="append",
        index=False,
        chunksize=5_000
    )

    conn.commit()

    # --------------------------------------------------
    # 8. Progress
    # --------------------------------------------------
    rows_in_chunk = len(chunk)
    s2_rows_processed += rows_in_chunk

    elapsed = time.time() - s2_start_time
    rate = s2_rows_processed / elapsed

    print(
        f"Chunk {s2_chunk_number:>3} | "
        f"Processed {s2_rows_processed:>10,} rows | "
        f"{rate:,.0f} rows/sec | "
        f"Chunk time: {time.time() - chunk_start:.1f}s"
    )

    # --------------------------------------------------
    # 9. Free RAM
    # --------------------------------------------------
    del db_chunk
    del chunk

    gc.collect()

print()
print("S2 processing completed.")

total_s2 = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
WHERE source = 'S2'
""").fetchone()[0]

print(f"S2 rows in SQLite: {total_s2:,}")

Starting full S2 processing...
Chunk size: 50,000

Chunk   1 | Processed     50,000 rows | 26,037 rows/sec | Chunk time: 1.8s
Chunk   2 | Processed    100,000 rows | 18,513 rows/sec | Chunk time: 3.3s
Chunk   3 | Processed    150,000 rows | 16,151 rows/sec | Chunk time: 3.6s
Chunk   4 | Processed    200,000 rows | 15,407 rows/sec | Chunk time: 3.5s
Chunk   5 | Processed    250,000 rows | 14,535 rows/sec | Chunk time: 4.0s
Chunk   6 | Processed    300,000 rows | 13,672 rows/sec | Chunk time: 4.6s
Chunk   7 | Processed    350,000 rows | 13,263 rows/sec | Chunk time: 4.2s
Chunk   8 | Processed    400,000 rows | 12,708 rows/sec | Chunk time: 4.9s
Chunk   9 | Processed    450,000 rows | 12,443 rows/sec | Chunk time: 4.5s
Chunk  10 | Processed    500,000 rows | 12,027 rows/sec | Chunk time: 5.2s
Chunk  11 | Processed    550,000 rows | 11,771 rows/sec | Chunk time: 4.9s
Chunk  12 | Processed    600,000 rows | 11,527 rows/sec | Chunk time: 5.1s
Chunk  13 | Processed    650,000 rows | 11,336 ro

In [42]:
# Check total rows
total_rows = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
""").fetchone()[0]

# Check S2 rows
s2_rows = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
WHERE source = 'S2'
""").fetchone()[0]

# Check unique S2 entity IDs
s2_unique_ids = conn.execute("""
SELECT COUNT(DISTINCT entity_id)
FROM entity_blocks
WHERE source = 'S2'
""").fetchone()[0]

print(f"Total rows in database:       {total_rows:,}")
print(f"S2 rows:                      {s2_rows:,}")
print(f"Unique S2 entity IDs:         {s2_unique_ids:,}")

Total rows in database:       5,034,616
S2 rows:                      5,034,616
Unique S2 entity IDs:         5,034,616


In [44]:
s3_path = os.path.join(
    TRAIN_PATH,
    "train_source3.tsv"
)

print("S3 path:")
print(s3_path)

print("\nFile exists:", os.path.exists(s3_path))

S3 path:
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\data\raw\train\train_source3.tsv

File exists: True


In [45]:
import time
import gc

S3_CHUNK_SIZE = 50_000

s3_start_time = time.time()
s3_rows_processed = 0
s3_chunk_number = 0

print("Starting full S3 processing...")
print(f"Chunk size: {S3_CHUNK_SIZE:,}")
print()

for chunk in pd.read_csv(
    s3_path,
    sep="\t",
    chunksize=S3_CHUNK_SIZE
):
    s3_chunk_number += 1
    chunk_start = time.time()

    # --------------------------------------------------
    # 1. Normalize name and address
    # --------------------------------------------------
    chunk["name_norm"] = chunk["business_name"].map(normalize_text)
    chunk["address_norm"] = chunk["business_address"].map(normalize_text)

    # --------------------------------------------------
    # 2. Name block
    # country + first 4 compact name characters
    # --------------------------------------------------
    chunk["name_compact"] = (
        chunk["name_norm"]
        .str.replace(" ", "", regex=False)
    )

    chunk["name_prefix"] = (
        chunk["name_compact"].str[:4]
    )

    chunk["block_key"] = (
        chunk["country"].fillna("").astype(str)
        + "_"
        + chunk["name_prefix"].fillna("")
    )

    # --------------------------------------------------
    # 3. Address prefix block
    # first 8 compact address characters
    # --------------------------------------------------
    chunk["address_compact"] = (
        chunk["address_norm"]
        .str.replace(" ", "", regex=False)
    )

    chunk["address_prefix"] = (
        chunk["address_compact"].str[:8]
    )

    # --------------------------------------------------
    # 4. First two address numbers
    # --------------------------------------------------
    chunk["address_numbers"] = (
        chunk["business_address"]
        .map(extract_address_numbers)
    )

    # --------------------------------------------------
    # 5. First address number block
    # country + first address number
    # --------------------------------------------------
    chunk["first_address_number"] = (
        chunk["business_address"]
        .map(extract_first_address_number)
    )

    chunk["number_block"] = (
        chunk["country"].fillna("").astype(str)
        + "_"
        + chunk["first_address_number"].fillna("")
    )

    # Never use empty-number blocks such as US_ or India_
    chunk.loc[
        chunk["first_address_number"].eq(""),
        "number_block"
    ] = ""

    # --------------------------------------------------
    # 6. Keep only database columns
    # --------------------------------------------------
    db_chunk = chunk[
        [
            "entity_id",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ].copy()

    db_chunk["source"] = "S3"

    db_chunk = db_chunk[
        [
            "entity_id",
            "source",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ]

    # --------------------------------------------------
    # 7. Write to SQLite
    # --------------------------------------------------
    db_chunk.to_sql(
        "entity_blocks",
        conn,
        if_exists="append",
        index=False,
        chunksize=5_000
    )

    conn.commit()

    # --------------------------------------------------
    # 8. Progress
    # --------------------------------------------------
    rows_in_chunk = len(chunk)
    s3_rows_processed += rows_in_chunk

    elapsed = time.time() - s3_start_time
    rate = s3_rows_processed / elapsed

    print(
        f"Chunk {s3_chunk_number:>3} | "
        f"Processed {s3_rows_processed:>10,} rows | "
        f"{rate:,.0f} rows/sec | "
        f"Chunk time: {time.time() - chunk_start:.1f}s"
    )

    # --------------------------------------------------
    # 9. Free RAM
    # --------------------------------------------------
    del db_chunk
    del chunk

    gc.collect()

print()
print("S3 processing completed.")

total_s3 = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
WHERE source = 'S3'
""").fetchone()[0]

print(f"S3 rows in SQLite: {total_s3:,}")

Starting full S3 processing...
Chunk size: 50,000

Chunk   1 | Processed     50,000 rows | 5,456 rows/sec | Chunk time: 9.0s
Chunk   2 | Processed    100,000 rows | 5,246 rows/sec | Chunk time: 9.7s
Chunk   3 | Processed    150,000 rows | 4,899 rows/sec | Chunk time: 11.3s
Chunk   4 | Processed    200,000 rows | 4,707 rows/sec | Chunk time: 11.7s
Chunk   5 | Processed    250,000 rows | 4,536 rows/sec | Chunk time: 12.4s
Chunk   6 | Processed    300,000 rows | 4,440 rows/sec | Chunk time: 12.2s
Chunk   7 | Processed    350,000 rows | 4,434 rows/sec | Chunk time: 11.1s
Chunk   8 | Processed    400,000 rows | 4,430 rows/sec | Chunk time: 11.1s
Chunk   9 | Processed    450,000 rows | 4,399 rows/sec | Chunk time: 11.8s
Chunk  10 | Processed    500,000 rows | 4,357 rows/sec | Chunk time: 12.2s
Chunk  11 | Processed    550,000 rows | 4,376 rows/sec | Chunk time: 10.7s
Chunk  12 | Processed    600,000 rows | 4,345 rows/sec | Chunk time: 12.2s
Chunk  13 | Processed    650,000 rows | 4,357 rows/

In [46]:
# Verify SQLite database integrity

total_rows = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
""").fetchone()[0]

s2_rows = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
WHERE source = 'S2'
""").fetchone()[0]

s3_rows = conn.execute("""
SELECT COUNT(*)
FROM entity_blocks
WHERE source = 'S3'
""").fetchone()[0]

s2_unique_ids = conn.execute("""
SELECT COUNT(DISTINCT entity_id)
FROM entity_blocks
WHERE source = 'S2'
""").fetchone()[0]

s3_unique_ids = conn.execute("""
SELECT COUNT(DISTINCT entity_id)
FROM entity_blocks
WHERE source = 'S3'
""").fetchone()[0]

print("SQLite verification")
print("-" * 40)
print(f"Total rows       : {total_rows:,}")
print(f"S2 rows          : {s2_rows:,}")
print(f"S3 rows          : {s3_rows:,}")
print(f"S2 unique IDs    : {s2_unique_ids:,}")
print(f"S3 unique IDs    : {s3_unique_ids:,}")

SQLite verification
----------------------------------------
Total rows       : 10,320,219
S2 rows          : 5,034,616
S3 rows          : 5,285,603
S2 unique IDs    : 5,034,616
S3 unique IDs    : 5,285,603


# step:6 Candidate Generation

In [47]:
# Test candidate generation for one S1 entity

test_row = train1.iloc[0]

print("S1 entity:")
print("Entity ID :", test_row["entity_id"])
print("Name      :", test_row["business_name"])
print("Country   :", test_row["country"])
print("Block key :", test_row["block_key"])
print("Addr pref :", test_row["address_prefix"])
print("Addr nums :", test_row["address_numbers"])
print("Num block :", test_row["number_block"])

S1 entity:
Entity ID : S1-925783039
Name      : Orelee's Barbershop
Country   : US
Block key : US_orel
Addr pref : 1795west
Addr nums : 1795
Num block : US_1795


In [48]:
# Test the 4 blocking rules for one S1 entity

test_row = train1.iloc[0]

queries = {
    "Name block": (
        "SELECT COUNT(*) FROM entity_blocks "
        "WHERE source IN ('S2', 'S3') "
        "AND block_key = ? "
        "AND block_key != ''",
        (test_row["block_key"],)
    ),

    "Address prefix": (
        "SELECT COUNT(*) FROM entity_blocks "
        "WHERE source IN ('S2', 'S3') "
        "AND address_prefix = ? "
        "AND address_prefix != ''",
        (test_row["address_prefix"],)
    ),

    "Address numbers": (
        "SELECT COUNT(*) FROM entity_blocks "
        "WHERE source IN ('S2', 'S3') "
        "AND address_numbers = ? "
        "AND address_numbers != ''",
        (test_row["address_numbers"],)
    ),

    "Number block": (
        "SELECT COUNT(*) FROM entity_blocks "
        "WHERE source IN ('S2', 'S3') "
        "AND number_block = ? "
        "AND number_block != ''",
        (test_row["number_block"],)
    )
}

print(f"S1 Entity: {test_row['entity_id']}")
print(f"Business : {test_row['business_name']}")
print()
print("Candidate counts")
print("-" * 40)

for name, (query, params) in queries.items():
    count = conn.execute(query, params).fetchone()[0]
    print(f"{name:<20}: {count:,}")

S1 Entity: S1-925783039
Business : Orelee's Barbershop

Candidate counts
----------------------------------------
Name block          : 953
Address prefix      : 3
Address numbers     : 211
Number block        : 292


In [49]:
# Retrieve unique candidate IDs for our test S1 entity

test_row = train1.iloc[0]

candidate_query = """
SELECT DISTINCT entity_id, source
FROM entity_blocks
WHERE source IN ('S2', 'S3')
AND (
       (block_key = ? AND block_key != '')
    OR (address_prefix = ? AND address_prefix != '')
    OR (address_numbers = ? AND address_numbers != '')
    OR (number_block = ? AND number_block != '')
)
"""

candidates = pd.read_sql_query(
    candidate_query,
    conn,
    params=(
        test_row["block_key"],
        test_row["address_prefix"],
        test_row["address_numbers"],
        test_row["number_block"]
    )
)

print(f"S1 Entity: {test_row['entity_id']}")
print(f"Total unique candidates: {len(candidates):,}")
print()

print("Candidates by source:")
print(candidates["source"].value_counts())

print()
print("First 20 candidates:")
print(candidates.head(20))

S1 Entity: S1-925783039
Total unique candidates: 1,251

Candidates by source:
source
S2    628
S3    623
Name: count, dtype: int64

First 20 candidates:
       entity_id source
0   S2-762023797     S2
1   S2-470535992     S2
2   S2-453851530     S2
3   S2-996204704     S2
4   S2-181058243     S2
5   S2-843675353     S2
6    S2-36816638     S2
7   S2-749547095     S2
8   S2-897924260     S2
9   S2-361957120     S2
10  S2-561926524     S2
11  S2-212566296     S2
12  S2-218978919     S2
13  S2-195915948     S2
14   S2-64372328     S2
15  S2-786711173     S2
16   S2-94796166     S2
17  S2-754814938     S2
18   S2-33501834     S2
19  S2-141689655     S2


In [50]:
# Check the ground-truth matches for our test S1 entity

s1_id = test_row["entity_id"]

gt_row = train_gr[
    train_gr["source1_entity_id"] == s1_id
]

print("S1 Entity:", s1_id)
print()

if len(gt_row) == 0:
    print("No ground-truth row found.")
else:
    matched_ids = gt_row.iloc[0]["matched_entity_ids"]

    if pd.isna(matched_ids):
        true_matches = []
    else:
        true_matches = [
            x.strip()
            for x in str(matched_ids).split(",")
            if x.strip()
        ]

    print(f"Ground-truth matches: {len(true_matches)}")
    print(true_matches)

    candidate_ids = set(candidates["entity_id"])

    found_matches = [
        x for x in true_matches
        if x in candidate_ids
    ]

    missed_matches = [
        x for x in true_matches
        if x not in candidate_ids
    ]

    print()
    print(f"Found in candidates : {len(found_matches)}")
    print(f"Missed              : {len(missed_matches)}")

    print()
    print("Found matches:")
    print(found_matches)

    print()
    print("Missed matches:")
    print(missed_matches)

S1 Entity: S1-925783039

Ground-truth matches: 4
['S2-157377754', 'S2-517291332', 'S3-997698194', 'S3-698172821']

Found in candidates : 4
Missed              : 0

Found matches:
['S2-157377754', 'S2-517291332', 'S3-997698194', 'S3-698172821']

Missed matches:
[]


In [51]:
# SAFE TEST: candidate generation for 100 S1 entities

import time

sample_s1 = train1.head(100).copy()

results = []

start_time = time.time()

for idx, row in sample_s1.iterrows():

    # Get candidates from the 4 blocking rules
    query = """
    SELECT DISTINCT entity_id, source
    FROM entity_blocks
    WHERE source IN ('S2', 'S3')
    AND (
           (block_key = ? AND block_key != '')
        OR (address_prefix = ? AND address_prefix != '')
        OR (address_numbers = ? AND address_numbers != '')
        OR (number_block = ? AND number_block != '')
    )
    """

    candidates_for_s1 = conn.execute(
        query,
        (
            row["block_key"],
            row["address_prefix"],
            row["address_numbers"],
            row["number_block"]
        )
    ).fetchall()

    candidate_ids = {x[0] for x in candidates_for_s1}

    # Ground truth
    gt_match = train_gr[
        train_gr["source1_entity_id"] == row["entity_id"]
    ]

    true_matches = set()

    if len(gt_match) > 0:
        matched_ids = gt_match.iloc[0]["matched_entity_ids"]

        if not pd.isna(matched_ids):
            true_matches = {
                x.strip()
                for x in str(matched_ids).split(",")
                if x.strip()
            }

    found = true_matches.intersection(candidate_ids)

    results.append({
        "s1_id": row["entity_id"],
        "candidate_count": len(candidate_ids),
        "true_match_count": len(true_matches),
        "found_match_count": len(found)
    })

elapsed = time.time() - start_time

sample_results = pd.DataFrame(results)

print("100-S1 blocking test completed")
print("-" * 45)
print(f"Time taken              : {elapsed:.2f} seconds")
print(f"Average candidates      : {sample_results['candidate_count'].mean():,.1f}")
print(f"Median candidates       : {sample_results['candidate_count'].median():,.0f}")
print(f"Maximum candidates      : {sample_results['candidate_count'].max():,}")
print(f"Total true matches      : {sample_results['true_match_count'].sum():,}")
print(f"Total recovered matches : {sample_results['found_match_count'].sum():,}")

total_true = sample_results["true_match_count"].sum()
total_found = sample_results["found_match_count"].sum()

if total_true > 0:
    print(f"Blocking recall         : {total_found / total_true:.4%}")
else:
    print("Blocking recall         : N/A")

100-S1 blocking test completed
---------------------------------------------
Time taken              : 41.50 seconds
Average candidates      : 23,364.2
Median candidates       : 11,597
Maximum candidates      : 174,673
Total true matches      : 322
Total recovered matches : 299
Blocking recall         : 92.8571%


In [52]:
# Analyze candidate volume contributed by each blocking rule
# SAFE: only 100 S1 rows

rule_counts = []

for _, row in sample_s1.iterrows():

    queries = {
        "name_block": (
            """
            SELECT COUNT(*)
            FROM entity_blocks
            WHERE source IN ('S2', 'S3')
              AND block_key = ?
              AND block_key != ''
            """,
            (row["block_key"],)
        ),

        "address_prefix": (
            """
            SELECT COUNT(*)
            FROM entity_blocks
            WHERE source IN ('S2', 'S3')
              AND address_prefix = ?
              AND address_prefix != ''
            """,
            (row["address_prefix"],)
        ),

        "address_numbers": (
            """
            SELECT COUNT(*)
            FROM entity_blocks
            WHERE source IN ('S2', 'S3')
              AND address_numbers = ?
              AND address_numbers != ''
            """,
            (row["address_numbers"],)
        ),

        "number_block": (
            """
            SELECT COUNT(*)
            FROM entity_blocks
            WHERE source IN ('S2', 'S3')
              AND number_block = ?
              AND number_block != ''
            """,
            (row["number_block"],)
        )
    }

    result = {"s1_id": row["entity_id"]}

    for rule, (query, params) in queries.items():
        result[rule] = conn.execute(
            query,
            params
        ).fetchone()[0]

    rule_counts.append(result)

rule_counts_df = pd.DataFrame(rule_counts)

print("Candidate volume by blocking rule")
print("=" * 60)

for rule in [
    "name_block",
    "address_prefix",
    "address_numbers",
    "number_block"
]:
    print()
    print(rule)
    print(f"  Mean   : {rule_counts_df[rule].mean():,.1f}")
    print(f"  Median : {rule_counts_df[rule].median():,.0f}")
    print(f"  Max    : {rule_counts_df[rule].max():,}")

Candidate volume by blocking rule

name_block
  Mean   : 7,399.4
  Median : 3,433
  Max    : 61,993

address_prefix
  Mean   : 319.9
  Median : 2
  Max    : 9,297

address_numbers
  Mean   : 3,039.2
  Median : 269
  Max    : 36,287

number_block
  Mean   : 14,558.4
  Median : 1,950
  Max    : 166,885


In [53]:
# Check the distribution of block sizes in SQLite
# No candidate pairs are generated.

block_rules = {
    "name_block": "block_key",
    "address_prefix": "address_prefix",
    "address_numbers": "address_numbers",
    "number_block": "number_block"
}

for rule_name, column_name in block_rules.items():

    print()
    print("=" * 60)
    print(rule_name)

    query = f"""
    SELECT
        {column_name} AS block_value,
        COUNT(*) AS block_size
    FROM entity_blocks
    WHERE source IN ('S2', 'S3')
      AND {column_name} != ''
    GROUP BY {column_name}
    """

    block_stats = pd.read_sql_query(query, conn)

    print(f"Unique blocks : {len(block_stats):,}")
    print(f"Mean size     : {block_stats['block_size'].mean():,.2f}")
    print(f"Median size   : {block_stats['block_size'].median():,.0f}")
    print(f"Maximum size  : {block_stats['block_size'].max():,}")

    print("\nSize percentiles:")
    print(
        block_stats["block_size"]
        .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    )

    del block_stats
    gc.collect()


name_block
Unique blocks : 282,496
Mean size     : 36.53
Median size   : 2
Maximum size  : 86,579

Size percentiles:
0.50      2.0
0.75      6.0
0.90     19.0
0.95     57.0
0.99    486.0
Name: block_size, dtype: float64

address_prefix
Unique blocks : 4,413,216
Mean size     : 2.26
Median size   : 1
Maximum size  : 32,338

Size percentiles:
0.50     1.0
0.75     2.0
0.90     4.0
0.95     5.0
0.99    12.0
Name: block_size, dtype: float64

address_numbers
Unique blocks : 915,055
Mean size     : 10.23
Median size   : 1
Maximum size  : 54,339

Size percentiles:
0.50      1.0
0.75      3.0
0.90      7.0
0.95     15.0
0.99    115.0
Name: block_size, dtype: float64

number_block
Unique blocks : 101,851
Mean size     : 91.93
Median size   : 5
Maximum size  : 166,885

Size percentiles:
0.50       5.0
0.75      19.0
0.90      97.0
0.95     254.0
0.99    1382.0
Name: block_size, dtype: float64


In [54]:
# Create block-size tables inside SQLite.
# This does NOT generate candidate pairs.

print("Creating block-size tables...")

block_columns = [
    "block_key",
    "address_prefix",
    "address_numbers",
    "number_block"
]

for column in block_columns:

    table_name = f"{column}_sizes"

    conn.execute(f"""
        DROP TABLE IF EXISTS {table_name}
    """)

    conn.execute(f"""
        CREATE TABLE {table_name} AS
        SELECT
            {column} AS block_value,
            COUNT(*) AS block_size
        FROM entity_blocks
        WHERE source IN ('S2', 'S3')
          AND {column} != ''
        GROUP BY {column}
    """)

    conn.execute(f"""
        CREATE INDEX idx_{column}_sizes
        ON {table_name}(block_value)
    """)

    conn.commit()

    count = conn.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    print(f"{column:<20} {count:,} blocks")

print()
print("Block-size tables created successfully.")

Creating block-size tables...
block_key            282,496 blocks
address_prefix       4,413,216 blocks
address_numbers      915,055 blocks
number_block         101,851 blocks

Block-size tables created successfully.


In [55]:
# Quick verification of block-size tables

tables = [
    "block_key_sizes",
    "address_prefix_sizes",
    "address_numbers_sizes",
    "number_block_sizes"
]

print("Block-size table verification")
print("-" * 45)

for table in tables:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table:<25}: {count:,} blocks")

Block-size table verification
---------------------------------------------
block_key_sizes          : 282,496 blocks
address_prefix_sizes     : 4,413,216 blocks
address_numbers_sizes    : 915,055 blocks
number_block_sizes       : 101,851 blocks


In [58]:
# SAFE: recreate only 10,000 positive pairs from ground truth

gt_path = os.path.join(
    TRAIN_PATH,
    "train_ground_truth.tsv"
)

gt_sample = pd.read_csv(
    gt_path,
    sep="\t",
    nrows=20_000
)

print("Ground truth sample loaded.")
print("Rows:", len(gt_sample))
print("Columns:", gt_sample.columns.tolist())

# Convert matched_entity_ids into individual positive pairs
sample_pairs = []

for _, row in gt_sample.iterrows():

    s1_id = row["source1_entity_id"]
    matched = row["matched_entity_ids"]

    if pd.isna(matched):
        continue

    matched_ids = str(matched).split(",")

    for matched_id in matched_ids:
        sample_pairs.append({
            "source1_entity_id": s1_id,
            "matched_entity_id": matched_id,
            "target": 1
        })

        if len(sample_pairs) >= 10_000:
            break

    if len(sample_pairs) >= 10_000:
        break

test_pairs = pd.DataFrame(sample_pairs)

print()
print(f"Positive pairs created: {len(test_pairs):,}")
print()
print(test_pairs.head())

Ground truth sample loaded.
Rows: 20000
Columns: ['source1_entity_id', 'matched_entity_ids']

Positive pairs created: 10,000

  source1_entity_id matched_entity_id  target
0         S1-965667      S2-681193310       1
1         S1-965667      S2-743505751       1
2         S1-965667      S3-775321672       1
3         S1-965667       S3-11291185       1
4         S1-965667      S3-860443364       1


In [59]:
# SAFE: attach S1 and matched S2/S3 blocking information

# 1. Attach S1 blocking keys
s1_lookup = train1[
    [
        "entity_id",
        "block_key",
        "address_prefix",
        "address_numbers",
        "number_block"
    ]
].set_index("entity_id")

test_pairs = test_pairs.join(
    s1_lookup,
    on="source1_entity_id"
)

print("S1 blocking keys attached.")


# 2. Get unique matched entity IDs
matched_ids = test_pairs["matched_entity_id"].unique()

print(f"Unique matched entities: {len(matched_ids):,}")


# 3. Retrieve S2/S3 blocking information from SQLite
candidate_lookup = {}

for entity_id in matched_ids:

    row = conn.execute("""
        SELECT
            block_key,
            address_prefix,
            address_numbers,
            number_block
        FROM entity_blocks
        WHERE entity_id = ?
    """, (entity_id,)).fetchone()

    if row is not None:
        candidate_lookup[entity_id] = row


print(
    f"Blocking information retrieved for "
    f"{len(candidate_lookup):,} matched entities."
)


# 4. Attach matched-entity blocking keys
test_pairs["c_block_key"] = test_pairs["matched_entity_id"].map(
    lambda x: candidate_lookup.get(
        x, (None, None, None, None)
    )[0]
)

test_pairs["c_address_prefix"] = test_pairs["matched_entity_id"].map(
    lambda x: candidate_lookup.get(
        x, (None, None, None, None)
    )[1]
)

test_pairs["c_address_numbers"] = test_pairs["matched_entity_id"].map(
    lambda x: candidate_lookup.get(
        x, (None, None, None, None)
    )[2]
)

test_pairs["c_number_block"] = test_pairs["matched_entity_id"].map(
    lambda x: candidate_lookup.get(
        x, (None, None, None, None)
    )[3]
)


print()
print("Final test-pair columns:")
print(test_pairs.columns.tolist())

print()
print(test_pairs.head())

S1 blocking keys attached.
Unique matched entities: 10,000


KeyboardInterrupt: 

In [60]:
print("Current notebook is ready.")
print(f"S1 rows: {len(train1):,}")

# Check that SQLite connection is still usable
print(
    "SQLite rows:",
    conn.execute("SELECT COUNT(*) FROM entity_blocks").fetchone()[0]
)

Current notebook is ready.
S1 rows: 2,206,821
SQLite rows: 10320219


In [61]:
# Create a compact address signature on a small sample only

address_test = train1[
    ["entity_id", "business_address", "country"]
].head(10_000).copy()

address_test["address_norm_test"] = (
    address_test["business_address"]
    .map(normalize_text)
)

# Remove spaces
address_test["address_compact_test"] = (
    address_test["address_norm_test"]
    .str.replace(" ", "", regex=False)
)

# First 12 characters of normalized address
address_test["address_sig_test"] = (
    address_test["address_compact_test"].str[:12]
)

print("Address signature test completed.")
print()
print(
    address_test[
        [
            "business_address",
            "address_sig_test"
        ]
    ].head(10)
)

Address signature test completed.

                                    business_address address_sig_test
0             1795 Westchester Drive, High Point, NC     1795westches
1                    17560 Ellis Road, Tahlequah, OK     17560ellisro
2                1712 Montebello Avenue, Phoenix, AZ     1712montebel
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD     2100camerond
4  797, Lake Town Block A, Kolkata, Howrah, West ...     797laketownb
5                  OH, Columbus, 5559 Orville Avenue     ohcolumbus55
6  2505, Tower 1, Oakwood, Runwal Greens, Mulund ...     2505tower1oa
7       1111 Church Street, Unit 2007, Nashville, TN     1111churchst
8              337 Oakland Avenue, Michigan City, IN     337oaklandav
9  294 Meadowcreek Drive, Unit Unit 2, Village Of...     294meadowcre


In [62]:
# Only 100,000 S1 rows

address_sample = train1[
    ["entity_id", "business_address", "country"]
].head(100_000).copy()

address_sample["address_norm_test"] = (
    address_sample["business_address"]
    .map(normalize_text)
)

address_sample["address_sig_test"] = (
    address_sample["address_norm_test"]
    .str.replace(" ", "", regex=False)
    .str[:12]
)

# Empty signatures should not be used as blocking keys
address_sample.loc[
    address_sample["address_sig_test"].eq(""),
    "address_sig_test"
] = np.nan

print("Rows tested:", len(address_sample))
print(
    "Unique address signatures:",
    address_sample["address_sig_test"].nunique()
)

print()
print("Signature size distribution:")
print(
    address_sample["address_sig_test"]
    .value_counts()
    .describe()
)

print()
print("Largest signatures:")
print(
    address_sample["address_sig_test"]
    .value_counts()
    .head(10)
)

Rows tested: 100000
Unique address signatures: 95602

Signature size distribution:
count    95602.000000
mean         1.046003
std          0.865282
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        137.000000
Name: count, dtype: float64

Largest signatures:
address_sig_test
uttarpradesh    137
maharashtram     78
mumbaimahara     66
maharashtrap     56
punemaharash     53
newdelhidelh     51
delhinewdelh     50
karnatakaban     45
madhyaprades     42
bangalorekar     40
Name: count, dtype: int64


In [63]:
# SAFE TEST — combined address signature
# Only 100,000 S1 rows

address_sample = train1[
    [
        "entity_id",
        "business_address",
        "country",
        "first_address_number"
    ]
].head(100_000).copy()

# Normalize address
address_sample["address_norm_test"] = (
    address_sample["business_address"]
    .map(normalize_text)
)

# Remove spaces
address_sample["address_compact_test"] = (
    address_sample["address_norm_test"]
    .str.replace(" ", "", regex=False)
)

# Take characters after the first address number.
# We use the first 8 compact characters as an additional signal.
address_sample["address_text_sig"] = (
    address_sample["address_compact_test"].str[:8]
)

# Combine country + first address number + address text
address_sample["address_combined_sig"] = (
    address_sample["country"].fillna("").astype(str)
    + "_"
    + address_sample["first_address_number"].fillna("")
    + "_"
    + address_sample["address_text_sig"].fillna("")
)

# Do NOT use signatures without a number
address_sample.loc[
    address_sample["first_address_number"].eq(""),
    "address_combined_sig"
] = np.nan

signature_counts = (
    address_sample["address_combined_sig"]
    .value_counts()
)

print("Rows tested:", len(address_sample))
print(
    "Unique combined signatures:",
    address_sample["address_combined_sig"].nunique()
)

print()
print("Signature size distribution:")
print(signature_counts.describe())

print()
print("Largest signatures:")
print(signature_counts.head(10))

Rows tested: 100000
Unique combined signatures: 91239

Signature size distribution:
count    91239.000000
mean         1.058177
std          0.830392
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        121.000000
Name: count, dtype: float64

Largest signatures:
address_combined_sig
India_1_1stfloor      121
India_2_2ndfloor      113
India_3_3rdfloor       83
India_4_4thfloor       51
India_201_flatno20     41
India_101_flatno10     39
India_5_5thfloor       36
India_202_flatno20     28
India_401_flatno40     23
India_12_plotno12      22
Name: count, dtype: int64


In [64]:
import psutil

ram = psutil.virtual_memory()

print(f"RAM total     : {ram.total / 1024**3:.2f} GB")
print(f"RAM available : {ram.available / 1024**3:.2f} GB")
print(f"RAM used      : {ram.percent}%")

RAM total     : 15.63 GB
RAM available : 6.04 GB
RAM used      : 61.4%


In [65]:
indexes = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'index'
      AND tbl_name = 'entity_blocks'
""").fetchall()

print("Indexes on entity_blocks:")
for idx in indexes:
    print(" -", idx[0])

Indexes on entity_blocks:
 - idx_block_key
 - idx_address_prefix
 - idx_address_numbers
 - idx_number_block


In [66]:
# SAFE TEST — candidate generation for only 10 S1 entities

import time

# First 10 S1 entities
s1_test = train1.head(10)

# Load their ground-truth matches directly from the GT file
gt_test = pd.read_csv(
    gt_path,
    sep="\t"
)

# Only keep the 10 S1 IDs
gt_test = gt_test[
    gt_test["source1_entity_id"].isin(
        s1_test["entity_id"]
    )
].copy()

# Create GT lookup
gt_lookup_test = {}

for _, row in gt_test.iterrows():

    matched = row["matched_entity_ids"]

    if pd.isna(matched):
        gt_lookup_test[row["source1_entity_id"]] = set()
    else:
        gt_lookup_test[row["source1_entity_id"]] = set(
            str(matched).split(",")
        )

print("Testing candidate generation for 10 S1 entities...")
print()

results = []

start_time = time.time()

for _, s1 in s1_test.iterrows():

    s1_id = s1["entity_id"]

    candidate_ids = set()

    # ------------------------------------------------
    # 1. Name block
    # ------------------------------------------------
    if s1["block_key"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE block_key = ?
        """, (s1["block_key"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # ------------------------------------------------
    # 2. Address prefix
    # ------------------------------------------------
    if s1["address_prefix"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE address_prefix = ?
        """, (s1["address_prefix"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # ------------------------------------------------
    # 3. Address numbers
    # ------------------------------------------------
    if s1["address_numbers"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE address_numbers = ?
        """, (s1["address_numbers"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # ------------------------------------------------
    # 4. First address number
    # ------------------------------------------------
    if s1["number_block"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE number_block = ?
        """, (s1["number_block"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # Only S2/S3 are candidates
    candidate_ids = {
        x for x in candidate_ids
        if x.startswith("S2-") or x.startswith("S3-")
    }

    true_matches = gt_lookup_test.get(
        s1_id,
        set()
    )

    recovered = true_matches.intersection(candidate_ids)

    results.append({
        "s1_id": s1_id,
        "candidates": len(candidate_ids),
        "true_matches": len(true_matches),
        "recovered": len(recovered),
        "missed": len(true_matches - candidate_ids)
    })

elapsed = time.time() - start_time

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

print()
print(f"Time taken: {elapsed:.2f} seconds")
print(
    f"Total candidates: "
    f"{results_df['candidates'].sum():,}"
)
print(
    f"Total true matches: "
    f"{results_df['true_matches'].sum():,}"
)
print(
    f"Total recovered: "
    f"{results_df['recovered'].sum():,}"
)

Testing candidate generation for 10 S1 entities...

       s1_id  candidates  true_matches  recovered  missed
S1-925783039        1251             4          4       0
S1-773889195       31403             6          6       0
S1-377745466        1300             4          4       0
S1-133037285        9656             6          6       0
S1-755362802        1982             2          1       1
S1-851869949       11168             5          5       0
S1-785847572       12849             4          4       0
 S1-27541239        2662             4          4       0
S1-629417405        7750             7          7       0
 S1-22305073       22282             4          4       0

Time taken: 0.90 seconds
Total candidates: 102,303
Total true matches: 46
Total recovered: 45


In [67]:
# SAFE SCALE TEST — 1,000 S1 entities
# Candidates are NOT stored.

import time
import gc

s1_test_1000 = train1.head(1_000)

# Ground truth for these 1,000 S1 entities
gt_1000 = pd.read_csv(
    gt_path,
    sep="\t"
)

gt_1000 = gt_1000[
    gt_1000["source1_entity_id"].isin(
        s1_test_1000["entity_id"]
    )
].copy()

gt_lookup_1000 = {}

for _, row in gt_1000.iterrows():

    matched = row["matched_entity_ids"]

    if pd.isna(matched):
        gt_lookup_1000[row["source1_entity_id"]] = set()
    else:
        gt_lookup_1000[row["source1_entity_id"]] = set(
            str(matched).split(",")
        )

print("Testing 1,000 S1 entities...")
print()

candidate_counts = []
total_true = 0
total_recovered = 0
total_missed = 0

start_time = time.time()

for i, (_, s1) in enumerate(s1_test_1000.iterrows(), start=1):

    candidate_ids = set()

    # Name block
    if s1["block_key"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE block_key = ?
        """, (s1["block_key"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # Address prefix
    if s1["address_prefix"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE address_prefix = ?
        """, (s1["address_prefix"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # Address numbers
    if s1["address_numbers"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE address_numbers = ?
        """, (s1["address_numbers"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # First address number
    if s1["number_block"] != "":
        rows = conn.execute("""
            SELECT entity_id
            FROM entity_blocks
            WHERE number_block = ?
        """, (s1["number_block"],)).fetchall()

        candidate_ids.update(row[0] for row in rows)

    # Keep only S2/S3
    candidate_ids = {
        x for x in candidate_ids
        if x.startswith("S2-") or x.startswith("S3-")
    }

    true_matches = gt_lookup_1000.get(
        s1["entity_id"],
        set()
    )

    recovered = true_matches.intersection(candidate_ids)

    candidate_counts.append(len(candidate_ids))

    total_true += len(true_matches)
    total_recovered += len(recovered)
    total_missed += len(true_matches - candidate_ids)

    if i % 100 == 0:
        print(
            f"Processed {i:,}/1,000 | "
            f"Elapsed: {time.time() - start_time:.1f}s"
        )

elapsed = time.time() - start_time

candidate_counts = np.array(candidate_counts)

print()
print("=" * 55)
print("1,000-S1 BLOCKING TEST RESULTS")
print("=" * 55)

print(f"Time taken       : {elapsed:.2f} seconds")
print(f"Average candidates: {candidate_counts.mean():,.0f}")
print(f"Median candidates : {np.median(candidate_counts):,.0f}")
print(
    f"95th percentile  : "
    f"{np.percentile(candidate_counts, 95):,.0f}"
)
print(f"Maximum candidates: {candidate_counts.max():,}")

print()
print(f"Total true matches : {total_true:,}")
print(f"Recovered matches  : {total_recovered:,}")
print(f"Missed matches     : {total_missed:,}")

if total_true > 0:
    print(
        f"Blocking recall   : "
        f"{total_recovered / total_true * 100:.2f}%"
    )

print("=" * 55)

gc.collect()

Testing 1,000 S1 entities...

Processed 100/1,000 | Elapsed: 17.6s
Processed 200/1,000 | Elapsed: 35.0s
Processed 300/1,000 | Elapsed: 49.8s
Processed 400/1,000 | Elapsed: 69.6s
Processed 500/1,000 | Elapsed: 84.1s
Processed 600/1,000 | Elapsed: 101.8s
Processed 700/1,000 | Elapsed: 123.9s
Processed 800/1,000 | Elapsed: 140.9s
Processed 900/1,000 | Elapsed: 155.5s
Processed 1,000/1,000 | Elapsed: 170.9s

1,000-S1 BLOCKING TEST RESULTS
Time taken       : 170.87 seconds
Average candidates: 24,233
Median candidates : 11,415
95th percentile  : 107,202
Maximum candidates: 221,643

Total true matches : 3,517
Recovered matches  : 3,281
Missed matches     : 236
Blocking recall   : 93.29%


10

In [68]:
s1 = train1.iloc[0]

print("S1 entity:", s1["entity_id"])
print("Name block:", s1["block_key"])
print("Address prefix:", s1["address_prefix"])
print("Address numbers:", s1["address_numbers"])
print("Number block:", s1["number_block"])
print()

# Look up block sizes
checks = [
    ("block_key", "block_key_sizes", s1["block_key"]),
    ("address_prefix", "address_prefix_sizes", s1["address_prefix"]),
    ("address_numbers", "address_numbers_sizes", s1["address_numbers"]),
    ("number_block", "number_block_sizes", s1["number_block"]),
]

for rule, table, value in checks:

    if value == "":
        print(f"{rule:<20}: EMPTY → ignored")
        continue

    size = conn.execute(
        f"""
        SELECT block_size
        FROM {table}
        WHERE block_value = ?
        """,
        (value,)
    ).fetchone()

    if size is None:
        print(f"{rule:<20}: not found")
    else:
        print(
            f"{rule:<20}: "
            f"{size[0]:,} candidates"
        )

print()
print("Threshold:", 10)

S1 entity: S1-925783039
Name block: US_orel
Address prefix: 1795west
Address numbers: 1795
Number block: US_1795

block_key           : 953 candidates
address_prefix      : 3 candidates
address_numbers     : 211 candidates
number_block        : 292 candidates

Threshold: 10


In [69]:
block_key_sizes_df = pd.read_sql_query("""
    SELECT block_value, block_size
    FROM block_key_sizes
""", conn)

print(
    f"Loaded block_key sizes: "
    f"{len(block_key_sizes_df):,} blocks"
)

print()
print(block_key_sizes_df.head())

Loaded block_key sizes: 282,496 blocks

  block_value  block_size
0      India_           7
1  India_0111           5
2  India_015t           1
3  India_017m           1
4  India_01st           1


In [70]:
sample_1000 = train1.head(1000).copy()

sample_1000 = sample_1000.merge(
    block_key_sizes_df,
    left_on="block_key",
    right_on="block_value",
    how="left"
)

sample_1000 = sample_1000.rename(
    columns={"block_size": "name_block_size"}
)

sample_1000 = sample_1000.drop(columns=["block_value"])

print("Name-block size statistics for 1,000 S1 entities:")
print(sample_1000["name_block_size"].describe())

print()
print("Largest name blocks:")
print(
    sample_1000[
        ["entity_id", "business_name", "block_key", "name_block_size"]
    ]
    .sort_values("name_block_size", ascending=False)
    .head(20)
    .to_string(index=False)
)

Name-block size statistics for 1,000 S1 entities:
count     1000.000000
mean      6534.167000
std       9034.239751
min          1.000000
25%        295.750000
50%       2434.500000
75%       9902.500000
max      61993.000000
Name: name_block_size, dtype: float64

Largest name blocks:
   entity_id                           business_name  block_key  name_block_size
S1-166601186                  Pediatric Medicine LLC    US_pedi            61993
S1-511863452 Pediatric Dentistry Prime Center Center    US_pedi            61993
S1-746273151         Pediatric Dental Physicians Inc    US_pedi            61993
S1-881770296         Pediatric Dentistry Clinic, Inc    US_pedi            61993
S1-740254254       Pediatric Prairie Physicians, LLC    US_pedi            61993
 S1-17339733 Pediatric Dentistry Care Associates Inc    US_pedi            61993
S1-940115232                            Shriji Group India_shri            57078
S1-401836525                    Interstate Charities    US_inte   

In [71]:
thresholds = [10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000]

print("Name-block threshold analysis")
print("-" * 70)

for threshold in thresholds:
    eligible = sample_1000[
        sample_1000["name_block_size"] <= threshold
    ]

    total_candidates = eligible["name_block_size"].sum()
    entities_kept = len(eligible)
    entities_skipped = len(sample_1000) - entities_kept

    print(
        f"Threshold <= {threshold:5,} | "
        f"Entities kept: {entities_kept:4,} | "
        f"Skipped: {entities_skipped:4,} | "
        f"Candidate comparisons: {total_candidates:,.0f}"
    )

Name-block threshold analysis
----------------------------------------------------------------------
Threshold <=    10 | Entities kept:   58 | Skipped:  942 | Candidate comparisons: 291
Threshold <=    25 | Entities kept:   88 | Skipped:  912 | Candidate comparisons: 802
Threshold <=    50 | Entities kept:  117 | Skipped:  883 | Candidate comparisons: 1,930
Threshold <=   100 | Entities kept:  149 | Skipped:  851 | Candidate comparisons: 4,279
Threshold <=   250 | Entities kept:  234 | Skipped:  766 | Candidate comparisons: 18,543
Threshold <=   500 | Entities kept:  308 | Skipped:  692 | Candidate comparisons: 46,344
Threshold <= 1,000 | Entities kept:  391 | Skipped:  609 | Candidate comparisons: 106,019
Threshold <= 2,500 | Entities kept:  502 | Skipped:  498 | Candidate comparisons: 288,185
Threshold <= 5,000 | Entities kept:  566 | Skipped:  434 | Candidate comparisons: 519,903
Threshold <= 10,000 | Entities kept:  753 | Skipped:  247 | Candidate comparisons: 1,889,376


In [72]:
gt_path = os.path.join(
    TRAIN_PATH,
    "train_ground_truth.tsv"
)

sample_s1_ids = set(sample_1000["entity_id"])

gt_sample = []

for chunk in pd.read_csv(
    gt_path,
    sep="\t",
    dtype=str,
    chunksize=100_000
):
    matched = chunk[
        chunk["source1_entity_id"].isin(sample_s1_ids)
    ]

    if len(matched) > 0:
        gt_sample.append(matched)

gt_1000 = pd.concat(gt_sample, ignore_index=True)

print("GT rows found:", len(gt_1000))
print()
print(gt_1000.head())

GT rows found: 1000

  source1_entity_id                                 matched_entity_ids
0      S1-844896591                           S2-362218588,S3-11966366
1      S1-548116192                                                NaN
2      S1-378978603             S2-319300693,S3-948532405,S3-921016187
3      S1-447452795                          S2-995233298,S3-366674728
4      S1-401761505  S2-422370961,S2-415623223,S3-17464132,S3-16723...


In [73]:
positive_sample = []

for _, row in gt_1000.iterrows():
    s1_id = row["source1_entity_id"]
    matched = row["matched_entity_ids"]

    if pd.isna(matched):
        continue

    for matched_id in str(matched).split(","):
        positive_sample.append({
            "source1_entity_id": s1_id,
            "matched_entity_id": matched_id
        })

positive_sample = pd.DataFrame(positive_sample)

print("Positive pairs in 1,000-S1 sample:", len(positive_sample))
print()
print(positive_sample.head())

Positive pairs in 1,000-S1 sample: 3517

  source1_entity_id matched_entity_id
0      S1-844896591      S2-362218588
1      S1-844896591       S3-11966366
2      S1-378978603      S2-319300693
3      S1-378978603      S3-948532405
4      S1-378978603      S3-921016187


In [74]:
# Temporary table containing the 3,517 true matched IDs
conn.execute("DROP TABLE IF EXISTS temp_positive_ids")

conn.execute("""
    CREATE TEMP TABLE temp_positive_ids (
        entity_id TEXT PRIMARY KEY
    )
""")

target_ids = [
    (x,) for x in positive_sample["matched_entity_id"].unique()
]

conn.executemany(
    "INSERT INTO temp_positive_ids (entity_id) VALUES (?)",
    target_ids
)

conn.commit()

print("Unique true matched S2/S3 IDs:", len(target_ids))

# One SQL join instead of thousands of queries
positive_target_blocks = pd.read_sql_query("""
    SELECT
        eb.entity_id,
        eb.source,
        eb.block_key,
        eb.address_prefix,
        eb.address_numbers,
        eb.number_block
    FROM entity_blocks eb
    INNER JOIN temp_positive_ids t
        ON eb.entity_id = t.entity_id
""", conn)

print()
print("Retrieved rows:", len(positive_target_blocks))
print()
print(positive_target_blocks.head())

Unique true matched S2/S3 IDs: 3517

Retrieved rows: 3517

      entity_id source   block_key address_prefix address_numbers number_block
0  S2-340882825     S2     US_crys                                            
1   S2-30568957     S2  India_msuj       roomno64           6_404      India_6
2  S2-434618903     S2     US_firs       58countr              58        US_58
3   S2-97351951     S2     US_neil       200faber             200       US_200
4   S2-80171195     S2     US_bayi       downersg             820       US_820


In [75]:
# Get S1 block key for each positive pair
positive_with_s1 = positive_sample.merge(
    sample_1000[["entity_id", "block_key"]],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
).drop(columns=["entity_id"])

positive_with_s1 = positive_with_s1.rename(
    columns={"block_key": "s1_block_key"}
)

# Add target S2/S3 block key
positive_with_s1 = positive_with_s1.merge(
    positive_target_blocks[["entity_id", "block_key"]],
    left_on="matched_entity_id",
    right_on="entity_id",
    how="left"
).drop(columns=["entity_id"])

positive_with_s1 = positive_with_s1.rename(
    columns={"block_key": "target_block_key"}
)

# A positive pair is recovered by name blocking if both
# entities share the same block_key.
positive_with_s1["same_name_block"] = (
    positive_with_s1["s1_block_key"]
    == positive_with_s1["target_block_key"]
)

# Get block size from our lookup table
positive_with_s1 = positive_with_s1.merge(
    block_key_sizes_df,
    left_on="s1_block_key",
    right_on="block_value",
    how="left"
).drop(columns=["block_value"])

positive_with_s1 = positive_with_s1.rename(
    columns={"block_size": "name_block_size"}
)

print("Positive pairs:", len(positive_with_s1))
print(
    "Positive pairs sharing the same name block:",
    positive_with_s1["same_name_block"].sum()
)

print()
print("Name-block recall by threshold")
print("-" * 70)

thresholds = [10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000]

for threshold in thresholds:

    recovered = (
        positive_with_s1["same_name_block"]
        & (positive_with_s1["name_block_size"] <= threshold)
    ).sum()

    recall = recovered / len(positive_with_s1)

    print(
        f"Threshold <= {threshold:5,} | "
        f"Recovered: {recovered:4,} / {len(positive_with_s1):,} | "
        f"Recall: {recall:.2%}"
    )

Positive pairs: 3517
Positive pairs sharing the same name block: 2679

Name-block recall by threshold
----------------------------------------------------------------------
Threshold <=    10 | Recovered:  123 / 3,517 | Recall: 3.50%
Threshold <=    25 | Recovered:  188 / 3,517 | Recall: 5.35%
Threshold <=    50 | Recovered:  258 / 3,517 | Recall: 7.34%
Threshold <=   100 | Recovered:  358 / 3,517 | Recall: 10.18%
Threshold <=   250 | Recovered:  603 / 3,517 | Recall: 17.15%
Threshold <=   500 | Recovered:  783 / 3,517 | Recall: 22.26%
Threshold <= 1,000 | Recovered: 1,017 / 3,517 | Recall: 28.92%
Threshold <= 2,500 | Recovered: 1,349 / 3,517 | Recall: 38.36%
Threshold <= 5,000 | Recovered: 1,547 / 3,517 | Recall: 43.99%
Threshold <= 10,000 | Recovered: 1,951 / 3,517 | Recall: 55.47%


In [76]:
sample_10000 = train1[
    [
        "entity_id",
        "block_key",
        "address_prefix",
        "address_numbers",
        "number_block"
    ]
].head(10_000).copy()

print("Sample size:", len(sample_10000))
print()
print(sample_10000.head())
print()
print("Memory usage:")
print(
    f"{sample_10000.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Sample size: 10000

      entity_id   block_key address_prefix address_numbers number_block
0  S1-925783039     US_orel       1795west            1795      US_1795
1  S1-773889195     US_prim       17560ell           17560     US_17560
2  S1-377745466     US_bret       1712mont            1712      US_1712
3  S1-133037285     US_chri       2100came            2100      US_2100
4  S1-755362802  India_prab       797laket             797    India_797

Memory usage:
2.71 MB


In [77]:
# Create temporary table containing S1 blocking keys
conn.execute("DROP TABLE IF EXISTS temp_s1_blocks")

conn.execute("""
    CREATE TEMP TABLE temp_s1_blocks (
        entity_id TEXT PRIMARY KEY,
        block_key TEXT,
        address_prefix TEXT,
        address_numbers TEXT,
        number_block TEXT
    )
""")

# Insert 10,000 S1 rows
s1_block_rows = list(
    sample_10000[
        [
            "entity_id",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ].itertuples(index=False, name=None)
)

conn.executemany(
    """
    INSERT INTO temp_s1_blocks
    (entity_id, block_key, address_prefix, address_numbers, number_block)
    VALUES (?, ?, ?, ?, ?)
    """,
    s1_block_rows
)

conn.commit()

print("Inserted S1 rows:", len(s1_block_rows))

Inserted S1 rows: 10000


In [78]:
conn.execute("DROP TABLE IF EXISTS temp_s1_blocks")

conn.execute("""
    CREATE TEMP TABLE temp_s1_blocks (
        entity_id TEXT PRIMARY KEY,
        block_key TEXT,
        address_prefix TEXT,
        address_numbers TEXT,
        number_block TEXT
    )
""")

s1_block_rows = list(
    sample_10000[
        [
            "entity_id",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ].itertuples(index=False, name=None)
)

conn.executemany(
    """
    INSERT INTO temp_s1_blocks
    (entity_id, block_key, address_prefix, address_numbers, number_block)
    VALUES (?, ?, ?, ?, ?)
    """,
    s1_block_rows
)

conn.commit()

print("Inserted S1 rows:", len(s1_block_rows))

Inserted S1 rows: 10000


In [80]:
print("train1:", "train1" in globals())
print("train2:", "train2" in globals())
print("train3:", "train3" in globals())
print("train_gr:", "train_gr" in globals())
print("conn:", "conn" in globals())

if "train1" in globals():
    print("S1 shape:", train1.shape)

if "train_gr" in globals():
    print("GT shape:", train_gr.shape)

train1: True
train2: False
train3: False
train_gr: True
conn: True
S1 shape: (2206821, 9)
GT shape: (2206821, 2)


In [82]:
train1.head(3)

,entity_id,business_name,business_address,country,block_key,address_prefix,address_numbers,first_address_number,number_block
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,US_orel,1795west,1795,1795,US_1795
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,US_prim,17560ell,17560,17560,US_17560
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,US_bret,1712mont,1712,1712,US_1712


In [83]:
train_gr.head(3)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"


In [84]:
# Create positive pairs from ground truth
# We will use a manageable sample for the first ML model.

POSITIVE_TARGET = 25_000

positive_rows = []

for row in train_gr.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched):
        continue

    for matched_id in str(matched).split(","):
        matched_id = matched_id.strip()

        if matched_id:
            positive_rows.append((s1_id, matched_id, 1))

        if len(positive_rows) >= POSITIVE_TARGET:
            break

    if len(positive_rows) >= POSITIVE_TARGET:
        break

positive_df = pd.DataFrame(
    positive_rows,
    columns=["s1_id", "candidate_id", "target"]
)

print("Positive training pairs:", len(positive_df))
print(positive_df.head())

Positive training pairs: 25000
       s1_id  candidate_id  target
0  S1-965667  S2-681193310       1
1  S1-965667  S2-743505751       1
2  S1-965667  S3-775321672       1
3  S1-965667   S3-11291185       1
4  S1-965667  S3-860443364       1


In [86]:
positive_df.shape

(25000, 3)

In [87]:
# Generate hard negative pairs using indexed blocking lookups

NEGATIVE_TARGET = 25_000

# Ground-truth positives for fast exclusion
positive_lookup = {}

for row in positive_df.itertuples(index=False):
    positive_lookup.setdefault(row.s1_id, set()).add(row.candidate_id)

negative_rows = []

# Only process S1 IDs represented in our positive sample
s1_ids = positive_df["s1_id"].drop_duplicates().tolist()

for i, s1_id in enumerate(s1_ids):

    # Get S1 blocking keys
    s1_info = train1.loc[
        train1["entity_id"] == s1_id,
        ["block_key", "address_prefix", "address_numbers", "number_block"]
    ]

    if s1_info.empty:
        continue

    s1_info = s1_info.iloc[0]

    candidate_ids = set()

    # Query each indexed block separately
    for column in [
        "block_key",
        "address_prefix",
        "address_numbers",
        "number_block"
    ]:

        value = s1_info[column]

        # Never use empty blocking keys
        if pd.isna(value) or value == "":
            continue

        query = f"""
            SELECT entity_id
            FROM entity_blocks
            WHERE {column} = ?
              AND source IN ('S2', 'S3')
            LIMIT 200
        """

        rows = conn.execute(query, (value,)).fetchall()

        for r in rows:
            candidate_ids.add(r[0])

    # Remove known positives
    known_positive = positive_lookup.get(s1_id, set())

    candidate_ids.difference_update(known_positive)

    # Take hard negatives
    for candidate_id in candidate_ids:
        negative_rows.append((s1_id, candidate_id, 0))

        if len(negative_rows) >= NEGATIVE_TARGET:
            break

    if len(negative_rows) >= NEGATIVE_TARGET:
        break

    if (i + 1) % 100 == 0:
        print(
            f"Processed {i + 1:,} S1 IDs | "
            f"Negatives: {len(negative_rows):,}"
        )

negative_df = pd.DataFrame(
    negative_rows,
    columns=["s1_id", "candidate_id", "target"]
)

print("\nNegative training pairs:", len(negative_df))
print(negative_df.head())


Negative training pairs: 25000
       s1_id  candidate_id  target
0  S1-965667  S2-331481433       0
1  S1-965667  S2-167654358       0
2  S1-965667  S2-391004108       0
3  S1-965667  S2-623511128       0
4  S1-965667  S2-808728262       0


In [88]:
negative_df.shape

(25000, 3)

In [89]:
ml_pairs = pd.concat(
    [positive_df, negative_df],
    ignore_index=True
)

# Shuffle the dataset
ml_pairs = ml_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("ML dataset shape:", ml_pairs.shape)
print("\nTarget distribution:")
print(ml_pairs["target"].value_counts())

print("\nSample:")
display(ml_pairs.head())

ML dataset shape: (50000, 3)

Target distribution:
target
0    25000
1    25000
Name: count, dtype: int64

Sample:


,s1_id,candidate_id,target
0,S1-282467635,S2-758104705,0
1,S1-259255930,S3-165168999,1
2,S1-553375105,S3-150800481,1
3,S1-871058137,S3-304673439,1
4,S1-818149988,S2-205418614,0


In [91]:
# Retrieve only the S2/S3 records needed for our 50,000 ML pairs

candidate_ids = ml_pairs["candidate_id"].unique().tolist()

print("Unique candidate IDs:", len(candidate_ids))

# SQLite has no parameter limit for a query this large,
# so process IDs in chunks.
CHUNK_SIZE = 5000

candidate_parts = []

for start in range(0, len(candidate_ids), CHUNK_SIZE):

    chunk = candidate_ids[start:start + CHUNK_SIZE]

    placeholders = ",".join(["?"] * len(chunk))

    query = f"""
        SELECT
            entity_id,
            source,
            business_name,
            business_address,
            country
        FROM entity_blocks
        WHERE entity_id IN ({placeholders})
    """

    part = pd.read_sql_query(
        query,
        conn,
        params=chunk
    )

    candidate_parts.append(part)

    print(
        f"Retrieved {min(start + CHUNK_SIZE, len(candidate_ids)):,}"
        f"/{len(candidate_ids):,}"
    )

candidate_df = pd.concat(
    candidate_parts,
    ignore_index=True
)

del candidate_parts
gc.collect()

print("\nCandidate records retrieved:", len(candidate_df))
print(candidate_df.head())

Unique candidate IDs: 49236


DatabaseError: Execution failed on sql '
        SELECT
            entity_id,
            source,
            business_name,
            business_address,
            country
        FROM entity_blocks
        WHERE entity_id IN (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?

In [92]:
# Prepare S1 information for the 50,000 ML pairs

s1_features = train1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

s1_features = s1_features.rename(
    columns={"entity_id": "s1_id"}
)

# Keep only S1 IDs used in our ML dataset
s1_features = s1_features[
    s1_features["s1_id"].isin(ml_pairs["s1_id"])
].copy()

print("S1 records needed:", len(s1_features))
print(s1_features.head())

S1 records needed: 6852
             s1_id                   business_name  \
484   S1-378978603              QH Vendome, L.L.C.   
731   S1-844896591      Modern Packaging Global PC   
1114  S1-966027321      Probst & Duran Newhold LLC   
1156   S1-88817160  Natka Beagle Pinnacle Hologram   
1171  S1-674323954    Patriot First Publishing Inc   

                                       business_address country  
484                   Katy, TX, 21342 Bending Green Way      US  
731            8935 Georgetown Pike, Fairfax County, VA      US  
1114                        Boston, 31 Floyd Street, MA      US  
1156  1040 Cleveland Street, City Of Wisconsin Rapid...      US  
1171                132 Sugar Toms Lane, Oyster Bay, NY      US  


In [93]:
# Retrieve only the S2/S3 records needed for our 50,000 ML pairs

needed_candidate_ids = set(
    ml_pairs["candidate_id"].unique()
)

candidate_parts = []

for source, filename in [
    ("S2", "train_source2.tsv"),
    ("S3", "train_source3.tsv")
]:
    
    filepath = os.path.join(TRAIN_PATH, filename)

    print(f"\nReading {source}...")

    for chunk in pd.read_csv(
        filepath,
        sep="\t",
        chunksize=200_000
    ):
        
        # Keep only required candidate IDs
        part = chunk[
            chunk["entity_id"].isin(needed_candidate_ids)
        ].copy()

        if len(part) > 0:
            candidate_parts.append(part)

            # Remove found IDs so later chunks don't need to search them
            needed_candidate_ids.difference_update(
                part["entity_id"]
            )

            print(
                f"Found {len(part):,} records | "
                f"Remaining: {len(needed_candidate_ids):,}"
            )

        # Stop once every required ID has been found
        if not needed_candidate_ids:
            break

    print(f"{source} finished.")

candidate_features = pd.concat(
    candidate_parts,
    ignore_index=True
)

del candidate_parts
gc.collect()

print("\nCandidate records retrieved:", len(candidate_features))
print("Still missing:", len(needed_candidate_ids))

print("\nColumns:")
print(candidate_features.columns.tolist())

display(candidate_features.head())


Reading S2...
Found 8,827 records | Remaining: 40,409
Found 3,379 records | Remaining: 37,030
Found 1,849 records | Remaining: 35,181
Found 1,554 records | Remaining: 33,627
Found 1,365 records | Remaining: 32,262
Found 1,332 records | Remaining: 30,930
Found 1,182 records | Remaining: 29,748
Found 1,094 records | Remaining: 28,654
Found 1,054 records | Remaining: 27,600
Found 984 records | Remaining: 26,616
Found 929 records | Remaining: 25,687
Found 882 records | Remaining: 24,805
Found 861 records | Remaining: 23,944
Found 769 records | Remaining: 23,175
Found 834 records | Remaining: 22,341
Found 754 records | Remaining: 21,587
Found 841 records | Remaining: 20,746
Found 772 records | Remaining: 19,974
Found 743 records | Remaining: 19,231
Found 708 records | Remaining: 18,523
Found 665 records | Remaining: 17,858
Found 649 records | Remaining: 17,209
Found 662 records | Remaining: 16,547
Found 591 records | Remaining: 15,956
Found 611 records | Remaining: 15,345
Found 135 records

,entity_id,business_name,business_address,country
0,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
1,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
2,S2-566688803,रियल मॉडर्न फूड लिमिटेड,"SHP NO. G-4, PLOT NO. 19, SECTOR 42, NERUL, TH...",India
3,S2-673704658,Behavioral Health Anchor Care,"1715-1719 Orchard Blossom Drive, CITY OF APPLE...",US
4,S2-49714549,CLASSIC [TECHNOLOGIES],"Rajasthan, HN G-103 113, VRINDAVAN VIHAR, DCM,...",India


In [94]:
# Merge S1 and candidate information into the ML pair dataset

feature_df = ml_pairs.merge(
    s1_features,
    on="s1_id",
    how="left"
)

feature_df = feature_df.merge(
    candidate_features,
    left_on="candidate_id",
    right_on="entity_id",
    how="left",
    suffixes=("_s1", "_candidate")
)

print("Feature dataset shape:", feature_df.shape)

print("\nMissing values:")
print(
    feature_df[
        [
            "business_name_s1",
            "business_name_candidate",
            "business_address_s1",
            "business_address_candidate",
            "country_s1",
            "country_candidate"
        ]
    ].isna().sum()
)

display(
    feature_df[
        [
            "s1_id",
            "candidate_id",
            "business_name_s1",
            "business_name_candidate",
            "country_s1",
            "country_candidate",
            "target"
        ]
    ].head()
)

Feature dataset shape: (50000, 10)

Missing values:
business_name_s1                 0
business_name_candidate          1
business_address_s1              0
business_address_candidate    1498
country_s1                       0
country_candidate                0
dtype: int64


,s1_id,candidate_id,business_name_s1,business_name_candidate,country_s1,country_candidate,target
0,S1-282467635,S2-758104705,Prem & Sons Pvt Ltd,બેસ્ટ કન્સ્ટ્રક્શન્સ પ્રાઇવેટ લિમિટેડ,India,India,0
1,S1-259255930,S3-165168999,FC Bluerock PC,FC Blenrck PC,US,US,1
2,S1-553375105,S3-150800481,"Kozlowski, Lopez & Smith LLC",lopezkozlowski.com,US,US,1
3,S1-871058137,S3-304673439,United Horizon Village Inc,United Horizon Village,US,US,1
4,S1-818149988,S2-205418614,Vadyne Inc,VADYNE,US,US,0


In [95]:
# Normalize business names and addresses

feature_df["name_s1_norm"] = feature_df["business_name_s1"].apply(normalize_text)
feature_df["name_candidate_norm"] = feature_df["business_name_candidate"].apply(normalize_text)

feature_df["address_s1_norm"] = feature_df["business_address_s1"].apply(normalize_text)
feature_df["address_candidate_norm"] = feature_df["business_address_candidate"].apply(normalize_text)

print("Normalization completed.")

display(
    feature_df[
        [
            "business_name_s1",
            "business_name_candidate",
            "name_s1_norm",
            "name_candidate_norm",
            "business_address_s1",
            "business_address_candidate",
            "address_s1_norm",
            "address_candidate_norm"
        ]
    ].head()
)

Normalization completed.


,business_name_s1,business_name_candidate,name_s1_norm,name_candidate_norm,business_address_s1,business_address_candidate,address_s1_norm,address_candidate_norm
0,Prem & Sons Pvt Ltd,બેસ્ટ કન્સ્ટ્રક્શન્સ પ્રાઇવેટ લિમિટેડ,prem sons pvt ltd,બેસ્ટ કન્સ્ટ્રક્શન્સ પ્રાઇવેટ લિમિટેડ,"D-4, Chandana Apartments82, Infantry Road, Ban...","BLOCK A 4TH FLOOR 419, KALP BUSINESS PARK OPP ...",d 4 chandana apartments82 infantry road bangal...,block a 4th floor 419 kalp business park opp v...
1,FC Bluerock PC,FC Blenrck PC,fc bluerock pc,fc blenrck pc,"AZ, Unit 2088, Chandler, 2022 Nevada Street","022 Nevada Street, Unit 2088, Chandler, Arizona",az unit 2088 chandler 2022 nevada street,022 nevada street unit 2088 chandler arizona
2,"Kozlowski, Lopez & Smith LLC",lopezkozlowski.com,kozlowski lopez smith llc,lopezkozlowski com,"5842 Baileys Ridge Drive, New Kent County, VA","5842-D Baileys Ridge Drive, Providence Forge, ...",5842 baileys ridge drive new kent county va,5842 d baileys ridge drive providence forge vi...
3,United Horizon Village Inc,United Horizon Village,united horizon village inc,united horizon village,"4848 Napoleon Street, Huntsville, OH","Huntsville, Ohio, 4848 Napoleon St",4848 napoleon street huntsville oh,huntsville ohio 4848 napoleon st
4,Vadyne Inc,VADYNE,vadyne inc,vadyne,"Greensboro, 1604 Washington Street, NC","6 SKYLINE WAY, HARTFORD, VT",greensboro 1604 washington street nc,6 skyline way hartford vt


In [96]:
try:
    from rapidfuzz import fuzz
    print("rapidfuzz is available")
except ImportError:
    print("rapidfuzz is NOT installed")

rapidfuzz is NOT installed


In [97]:
from difflib import SequenceMatcher

def similarity_ratio(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio() * 100


def token_sort_similarity(a, b):
    a_tokens = str(a).split()
    b_tokens = str(b).split()

    a_sorted = " ".join(sorted(a_tokens))
    b_sorted = " ".join(sorted(b_tokens))

    return SequenceMatcher(
        None,
        a_sorted,
        b_sorted
    ).ratio() * 100


# Name similarity features
feature_df["name_ratio"] = [
    similarity_ratio(a, b)
    for a, b in zip(
        feature_df["name_s1_norm"],
        feature_df["name_candidate_norm"]
    )
]

feature_df["name_token_sort_ratio"] = [
    token_sort_similarity(a, b)
    for a, b in zip(
        feature_df["name_s1_norm"],
        feature_df["name_candidate_norm"]
    )
]


# Address similarity features
feature_df["address_ratio"] = [
    similarity_ratio(a, b)
    for a, b in zip(
        feature_df["address_s1_norm"],
        feature_df["address_candidate_norm"]
    )
]

feature_df["address_token_sort_ratio"] = [
    token_sort_similarity(a, b)
    for a, b in zip(
        feature_df["address_s1_norm"],
        feature_df["address_candidate_norm"]
    )
]


print("Similarity features created.")

display(
    feature_df[
        [
            "name_ratio",
            "name_token_sort_ratio",
            "address_ratio",
            "address_token_sort_ratio",
            "target"
        ]
    ].head(10)
)

Similarity features created.


,name_ratio,name_token_sort_ratio,address_ratio,address_token_sort_ratio,target
0,11.111111,11.111111,32.298137,17.391304,0
1,88.888889,88.888889,54.761905,92.857143,1
2,55.813953,41.860465,67.368421,67.368421,1
3,91.666667,91.666667,48.484848,90.909091,1
4,75.000000,75.000000,29.508197,39.344262,0
5,48.275862,48.275862,24.817518,32.116788,0
6,93.023256,93.023256,71.428571,64.285714,1
7,21.621622,16.216216,47.761194,47.761194,0
8,9.836066,9.836066,95.734597,95.734597,1
9,7.547170,7.547170,39.694656,65.648855,0


In [98]:
feature_df.shape

(50000, 18)

In [100]:
feature_df.head()

,s1_id,candidate_id,target,business_name_s1,business_address_s1,country_s1,entity_id,business_name_candidate,business_address_candidate,country_candidate,name_s1_norm,name_candidate_norm,address_s1_norm,address_candidate_norm,name_ratio,name_token_sort_ratio,address_ratio,address_token_sort_ratio
0,S1-282467635,S2-758104705,0,Prem & Sons Pvt Ltd,"D-4, Chandana Apartments82, Infantry Road, Ban...",India,S2-758104705,બેસ્ટ કન્સ્ટ્રક્શન્સ પ્રાઇવેટ લિમિટેડ,"BLOCK A 4TH FLOOR 419, KALP BUSINESS PARK OPP ...",India,prem sons pvt ltd,બેસ્ટ કન્સ્ટ્રક્શન્સ પ્રાઇવેટ લિમિટેડ,d 4 chandana apartments82 infantry road bangal...,block a 4th floor 419 kalp business park opp v...,11.111111,11.111111,32.298137,17.391304
1,S1-259255930,S3-165168999,1,FC Bluerock PC,"AZ, Unit 2088, Chandler, 2022 Nevada Street",US,S3-165168999,FC Blenrck PC,"022 Nevada Street, Unit 2088, Chandler, Arizona",US,fc bluerock pc,fc blenrck pc,az unit 2088 chandler 2022 nevada street,022 nevada street unit 2088 chandler arizona,88.888889,88.888889,54.761905,92.857143
2,S1-553375105,S3-150800481,1,"Kozlowski, Lopez & Smith LLC","5842 Baileys Ridge Drive, New Kent County, VA",US,S3-150800481,lopezkozlowski.com,"5842-D Baileys Ridge Drive, Providence Forge, ...",US,kozlowski lopez smith llc,lopezkozlowski com,5842 baileys ridge drive new kent county va,5842 d baileys ridge drive providence forge vi...,55.813953,41.860465,67.368421,67.368421
3,S1-871058137,S3-304673439,1,United Horizon Village Inc,"4848 Napoleon Street, Huntsville, OH",US,S3-304673439,United Horizon Village,"Huntsville, Ohio, 4848 Napoleon St",US,united horizon village inc,united horizon village,4848 napoleon street huntsville oh,huntsville ohio 4848 napoleon st,91.666667,91.666667,48.484848,90.909091
4,S1-818149988,S2-205418614,0,Vadyne Inc,"Greensboro, 1604 Washington Street, NC",US,S2-205418614,VADYNE,"6 SKYLINE WAY, HARTFORD, VT",US,vadyne inc,vadyne,greensboro 1604 washington street nc,6 skyline way hartford vt,75.000000,75.000000,29.508197,39.344262


In [101]:
# ---------------------------------------------------------
# Additional entity-resolution features
# ---------------------------------------------------------

# 1. Country match
feature_df["country_match"] = (
    feature_df["country_s1"].fillna("").str.lower()
    ==
    feature_df["country_candidate"].fillna("").str.lower()
).astype(int)


# 2. Exact name match
feature_df["name_exact_match"] = (
    feature_df["name_s1_norm"]
    ==
    feature_df["name_candidate_norm"]
).astype(int)


# 3. Exact address match
feature_df["address_exact_match"] = (
    feature_df["address_s1_norm"]
    ==
    feature_df["address_candidate_norm"]
).astype(int)


# 4. Name containment
# True if one normalized name is contained inside the other
feature_df["name_contains"] = (
    feature_df.apply(
        lambda row:
        int(
            row["name_s1_norm"] != ""
            and row["name_candidate_norm"] != ""
            and (
                row["name_s1_norm"] in row["name_candidate_norm"]
                or
                row["name_candidate_norm"] in row["name_s1_norm"]
            )
        ),
        axis=1
    )
)


# 5. Address containment
feature_df["address_contains"] = (
    feature_df.apply(
        lambda row:
        int(
            row["address_s1_norm"] != ""
            and row["address_candidate_norm"] != ""
            and (
                row["address_s1_norm"] in row["address_candidate_norm"]
                or
                row["address_candidate_norm"] in row["address_s1_norm"]
            )
        ),
        axis=1
    )
)


# 6. Name length difference
feature_df["name_length_diff"] = (
    feature_df["name_s1_norm"].str.len()
    -
    feature_df["name_candidate_norm"].str.len()
).abs()


# 7. Address length difference
feature_df["address_length_diff"] = (
    feature_df["address_s1_norm"].str.len()
    -
    feature_df["address_candidate_norm"].str.len()
).abs()


# 8. First address number match
def first_number(text):
    numbers = re.findall(r"\d+", str(text))
    return numbers[0] if numbers else ""


feature_df["first_number_s1"] = (
    feature_df["address_s1_norm"].apply(first_number)
)

feature_df["first_number_candidate"] = (
    feature_df["address_candidate_norm"].apply(first_number)
)

feature_df["first_number_match"] = (
    (feature_df["first_number_s1"] != "")
    &
    (feature_df["first_number_s1"] == feature_df["first_number_candidate"])
).astype(int)


print("Additional features created.")

display(
    feature_df[
        [
            "country_match",
            "name_exact_match",
            "address_exact_match",
            "name_contains",
            "address_contains",
            "name_length_diff",
            "address_length_diff",
            "first_number_match",
            "target"
        ]
    ].head(10)
)

Additional features created.


,country_match,name_exact_match,address_exact_match,name_contains,address_contains,name_length_diff,address_length_diff,first_number_match,target
0,1,0,0,0,0,20,43,1,0
1,1,0,0,0,0,1,4,0,1
2,1,0,0,0,0,7,9,1,1
3,1,0,0,1,0,4,2,1,1
4,1,0,0,1,0,4,11,0,0
5,1,0,0,0,0,3,5,0,0
6,1,0,0,1,0,3,4,1,1
7,1,0,0,0,0,7,17,1,0
8,1,0,0,0,0,1,9,1,1
9,1,0,0,0,0,7,27,1,0


In [102]:
feature_df.shape

(50000, 28)

In [103]:
feature_columns = [
    "name_ratio",
    "name_token_sort_ratio",
    "address_ratio",
    "address_token_sort_ratio",
    "country_match",
    "name_exact_match",
    "address_exact_match",
    "name_contains",
    "address_contains",
    "name_length_diff",
    "address_length_diff",
    "first_number_match"
]

X = feature_df[feature_columns].copy()
y = feature_df["target"].copy()

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Complete feature_df:", feature_df.shape)
print("Final X:", X.shape)
print("Final y:", y.shape)

print("\nX columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Complete feature_df: (50000, 28)
Final X: (50000, 12)
Final y: (50000,)

X columns:
['name_ratio', 'name_token_sort_ratio', 'address_ratio', 'address_token_sort_ratio', 'country_match', 'name_exact_match', 'address_exact_match', 'name_contains', 'address_contains', 'name_length_diff', 'address_length_diff', 'first_number_match']

Target distribution:
target
0    25000
1    25000
Name: count, dtype: int64


# Group-based train/validation split

In [104]:
from sklearn.model_selection import GroupShuffleSplit

groups = feature_df["s1_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()

y_train = y.iloc[train_idx].copy()
y_val = y.iloc[val_idx].copy()

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation set:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

# Verify that no S1 entity appears in both sets
train_s1 = set(feature_df.iloc[train_idx]["s1_id"])
val_s1 = set(feature_df.iloc[val_idx]["s1_id"])

overlap = train_s1.intersection(val_s1)

print("\nS1 entities in both train and validation:", len(overlap))

Training set:
X_train: (41523, 12)
y_train: (41523,)

Validation set:
X_val: (8477, 12)
y_val: (8477,)

Training target distribution:
target
0    21547
1    19976
Name: count, dtype: int64

Validation target distribution:
target
1    5024
0    3453
Name: count, dtype: int64

S1 entities in both train and validation: 0


# Train XGBoost

In [105]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("XGBoost training completed successfully.")

XGBoost training completed successfully.


In [106]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predicted probabilities
y_val_proba = model.predict_proba(X_val)[:, 1]

# Default threshold
y_val_pred = (y_val_proba >= 0.50).astype(int)

# Metrics
roc_auc = roc_auc_score(y_val, y_val_proba)
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("Validation Results")
print("=" * 40)
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Validation Results
ROC-AUC  : 0.9988
Accuracy : 0.9836
Precision: 0.9876
Recall   : 0.9847
F1 Score : 0.9861

Confusion Matrix:
[[3391   62]
 [  77 4947]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      3453
           1       0.99      0.98      0.99      5024

    accuracy                           0.98      8477
   macro avg       0.98      0.98      0.98      8477
weighted avg       0.98      0.98      0.98      8477



In [107]:
from sklearn.metrics import fbeta_score
import numpy as np
import pandas as pd

# Validation predictions
val_results = feature_df.iloc[val_idx][
    ["s1_id", "candidate_id", "target"]
].copy()

val_results["probability"] = y_val_proba

# Test several thresholds
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

scores = []

for threshold in thresholds:
    val_results["prediction"] = (
        val_results["probability"] >= threshold
    ).astype(int)

    per_s1_scores = []

    for s1_id, group in val_results.groupby("s1_id"):
        y_true = group["target"].values
        y_pred = group["prediction"].values

        score = fbeta_score(
            y_true,
            y_pred,
            beta=0.5,
            zero_division=0
        )

        per_s1_scores.append(score)

    scores.append({
        "threshold": threshold,
        "macro_f0.5": np.mean(per_s1_scores),
        "predicted_positive_pairs": val_results[
            "prediction"
        ].sum()
    })

threshold_results = pd.DataFrame(scores)

threshold_results.sort_values(
    "macro_f0.5",
    ascending=False
).reset_index(drop=True)

,threshold,macro_f0.5,predicted_positive_pairs
0,0.15,0.995558,5141
1,0.10,0.995434,5198
2,0.20,0.995278,5104
3,0.25,0.995072,5084
4,0.30,0.994896,5064
5,0.35,0.994800,5053
6,0.40,0.994505,5034
7,0.45,0.994111,5019
8,0.50,0.993910,5009
9,0.55,0.993852,5000


In [108]:
import os
import pandas as pd

test1_path = os.path.join(TEST_PATH, "test_source1.tsv")
test2_path = os.path.join(TEST_PATH, "test_source2.tsv")
test3_path = os.path.join(TEST_PATH, "test_source3.tsv")

test1_sample = pd.read_csv(test1_path, sep="\t", nrows=5)
test2_sample = pd.read_csv(test2_path, sep="\t", nrows=5)
test3_sample = pd.read_csv(test3_path, sep="\t", nrows=5)

print("TEST S1 columns:", test1_sample.columns.tolist())
print("TEST S2 columns:", test2_sample.columns.tolist())
print("TEST S3 columns:", test3_sample.columns.tolist())

print("\nFile sizes:")
for path in [test1_path, test2_path, test3_path]:
    print(
        os.path.basename(path),
        f"{os.path.getsize(path) / (1024**3):.2f} GB"
    )

TEST S1 columns: ['entity_id', 'business_name', 'business_address', 'country']
TEST S2 columns: ['entity_id', 'business_name', 'business_address', 'country']
TEST S3 columns: ['entity_id', 'business_name', 'business_address', 'country']

File sizes:
test_source1.tsv 0.16 GB
test_source2.tsv 0.47 GB
test_source3.tsv 0.47 GB


In [109]:
import sqlite3
import pandas as pd
import os
import re
import time
import gc

# Reuse our existing database
# conn should already exist
print("Using database:", DB_PATH)

# Remove test rows if this cell is ever rerun
conn.execute("""
    DELETE FROM entity_blocks
    WHERE source IN ('TEST2', 'TEST3')
""")
conn.commit()

# Drop indexes temporarily for faster bulk insertion
for idx in [
    "idx_block_key",
    "idx_address_prefix",
    "idx_address_numbers",
    "idx_number_block"
]:
    conn.execute(f"DROP INDEX IF EXISTS {idx}")

conn.commit()

def make_block_rows(chunk):
    # Normalize business name
    name_norm = (
        chunk["business_name"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Remove punctuation/symbols approximately for blocking
    name_norm = name_norm.str.replace(
        r"[^\w\s]",
        " ",
        regex=True
    ).str.replace(
        r"\s+",
        " ",
        regex=True
    ).str.strip()

    name_prefix = (
        name_norm
        .str.replace(" ", "", regex=False)
        .str[:4]
    )

    country = (
        chunk["country"]
        .fillna("")
        .astype(str)
    )

    block_key = country + "_" + name_prefix

    # Address normalization
    address_norm = (
        chunk["business_address"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    address_prefix = address_norm.str[:8]

    # First two numbers from address
    address_numbers = address_norm.apply(
        lambda x: "_".join(re.findall(r"\d+", x)[:2])
    )

    # First address number
    first_number = address_norm.apply(
        lambda x: (
            re.findall(r"\d+", x)[0]
            if re.findall(r"\d+", x)
            else ""
        )
    )

    number_block = country + "_" + first_number

    result = pd.DataFrame({
        "entity_id": chunk["entity_id"].astype(str),
        "source": source_name,
        "block_key": block_key,
        "address_prefix": address_prefix,
        "address_numbers": address_numbers,
        "number_block": number_block
    })

    return result


for test_path, source_name in [
    (test2_path, "TEST2"),
    (test3_path, "TEST3")
]:

    print("\n" + "=" * 60)
    print("Processing:", source_name)
    print("=" * 60)

    start_time = time.time()
    total_rows = 0
    chunk_no = 0

    for chunk in pd.read_csv(
        test_path,
        sep="\t",
        chunksize=200_000,
        dtype=str
    ):
        chunk_no += 1

        rows = make_block_rows(chunk)

        rows.to_sql(
            "entity_blocks",
            conn,
            if_exists="append",
            index=False
        )

        total_rows += len(rows)

        elapsed = time.time() - start_time

        print(
            f"{source_name} | "
            f"chunk {chunk_no} | "
            f"rows {total_rows:,} | "
            f"time {elapsed/60:.1f} min"
        )

        del chunk, rows
        gc.collect()

# Recreate indexes
print("\nCreating indexes...")

conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_block_key
    ON entity_blocks(block_key)
""")

conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_address_prefix
    ON entity_blocks(address_prefix)
""")

conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_address_numbers
    ON entity_blocks(address_numbers)
""")

conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_number_block
    ON entity_blocks(number_block)
""")

conn.commit()

print("\nTest blocking database ready.")

for source in ["TEST2", "TEST3"]:
    count = conn.execute(
        "SELECT COUNT(*) FROM entity_blocks WHERE source=?",
        (source,)
    ).fetchone()[0]

    print(f"{source}: {count:,} rows")

print(
    "Total test candidate records:",
    conn.execute("""
        SELECT COUNT(*)
        FROM entity_blocks
        WHERE source IN ('TEST2','TEST3')
    """).fetchone()[0]
)

Using database: C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\data\processed\blocking\entity_blocking.db

Processing: TEST2
TEST2 | chunk 1 | rows 200,000 | time 0.1 min
TEST2 | chunk 2 | rows 400,000 | time 0.2 min
TEST2 | chunk 3 | rows 600,000 | time 0.3 min
TEST2 | chunk 4 | rows 800,000 | time 0.4 min
TEST2 | chunk 5 | rows 1,000,000 | time 0.4 min
TEST2 | chunk 6 | rows 1,200,000 | time 0.5 min
TEST2 | chunk 7 | rows 1,400,000 | time 0.6 min
TEST2 | chunk 8 | rows 1,600,000 | time 0.7 min
TEST2 | chunk 9 | rows 1,800,000 | time 0.8 min
TEST2 | chunk 10 | rows 2,000,000 | time 0.9 min
TEST2 | chunk 11 | rows 2,200,000 | time 1.0 min
TEST2 | chunk 12 | rows 2,400,000 | time 1.1 min
TEST2 | chunk 13 | rows 2,600,000 | time 1.2 min
TEST2 | chunk 14 | rows 2,800,000 | time 1.3 min
TEST2 | chunk 15 | rows 3,000,000 | time 1.4 min
TEST2 | chunk 16 | rows 3,200,000 | time 1.4 min
TEST2 | chunk 17 | rows 3,400,000 | time 1.5 min
TEST2 | chunk 18 | rows 3,600,000 | time 1.6 min


In [110]:
# Check test S1 row count first

test1_count = sum(
    1 for _ in open(test1_path, "r", encoding="utf-8")
) - 1

print(f"Test S1 rows: {test1_count:,}")

Test S1 rows: 1,732,544


In [111]:
print("positive_pairs exists:", "positive_pairs" in globals())

if "positive_pairs" in globals():
    print("positive_pairs shape:", positive_pairs.shape)
    print(positive_pairs.head())

print("\nblock_key_sizes_df exists:", "block_key_sizes_df" in globals())

if "block_key_sizes_df" in globals():
    print("block_key_sizes_df shape:", block_key_sizes_df.shape)
    print(block_key_sizes_df.head())

positive_pairs exists: False

block_key_sizes_df exists: True
block_key_sizes_df shape: (282496, 2)
  block_value  block_size
0      India_           7
1  India_0111           5
2  India_015t           1
3  India_017m           1
4  India_01st           1


In [112]:
# Read only the columns needed for blocking
test1_blocks = pd.read_csv(
    test1_path,
    sep="\t",
    usecols=["entity_id", "business_name", "business_address", "country"],
    dtype=str
)

# Create blocking fields
test1_blocks["name_norm"] = (
    test1_blocks["business_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

test1_blocks["name_prefix"] = (
    test1_blocks["name_norm"]
    .str.replace(" ", "", regex=False)
    .str[:4]
)

test1_blocks["block_key"] = (
    test1_blocks["country"].fillna("").astype(str)
    + "_"
    + test1_blocks["name_prefix"]
)

test1_blocks["address_norm"] = (
    test1_blocks["business_address"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

test1_blocks["address_prefix"] = test1_blocks["address_norm"].str[:8]

test1_blocks["address_numbers"] = test1_blocks["address_norm"].apply(
    lambda x: "_".join(re.findall(r"\d+", x)[:2])
)

test1_blocks["first_number"] = test1_blocks["address_norm"].apply(
    lambda x: re.findall(r"\d+", x)[0]
    if re.findall(r"\d+", x)
    else ""
)

test1_blocks["number_block"] = (
    test1_blocks["country"].fillna("").astype(str)
    + "_"
    + test1_blocks["first_number"]
)

print("Test S1 loaded:", test1_blocks.shape)

print("\nLargest S1 name blocks:")
print(
    test1_blocks["block_key"]
    .value_counts()
    .head(20)
)

print("\nEmpty name block:", 
      (test1_blocks["block_key"] == "_").sum())

print("\nEmpty number blocks:",
      (test1_blocks["number_block"].str.endswith("_")).sum())

Test S1 loaded: (1732544, 12)

Largest S1 name blocks:
block_key
India_shiv     10407
India_indi      8012
US_pedi         7634
France_mais     7441
India_shre      7222
US_inte         6901
India_gold      6809
India_tech      6798
India_sout      6523
India_east      6384
France_bord     5208
France_étab     5174
France_cent     4820
France_écol     4595
India_mumb      4526
France_nant     4477
India_gree      4461
France_asso     4435
India_newd      4435
India_inte      4373
Name: count, dtype: int64

Empty name block: 0

Empty number blocks: 71838


In [113]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """,
    conn
)

print(tables)

                    name
0  address_numbers_sizes
1   address_prefix_sizes
2        block_key_sizes
3       candidates_10000
4          entity_blocks
5     number_block_sizes


In [114]:
print("Block size statistics\n")

for table in [
    "block_key_sizes",
    "address_prefix_sizes",
    "address_numbers_sizes",
    "number_block_sizes"
]:
    df = pd.read_sql_query(
        f"""
        SELECT block_size
        FROM {table}
        """,
        conn
    )

    print(f"\n{table}")
    print(df["block_size"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ))

    print(
        "Blocks <= 100:",
        (df["block_size"] <= 100).sum()
    )
    print(
        "Blocks <= 500:",
        (df["block_size"] <= 500).sum()
    )
    print(
        "Blocks <= 1000:",
        (df["block_size"] <= 1000).sum()
    )

Block size statistics


block_key_sizes
count    282496.000000
mean         36.532266
std         510.585789
min           1.000000
50%           2.000000
75%           6.000000
90%          19.000000
95%          57.000000
99%         486.000000
max       86579.000000
Name: block_size, dtype: float64
Blocks <= 100: 272251
Blocks <= 500: 279779
Blocks <= 1000: 281120

address_prefix_sizes
count    4.413216e+06
mean     2.260333e+00
std      3.381900e+01
min      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
90%      4.000000e+00
95%      5.000000e+00
99%      1.200000e+01
max      3.233800e+04
Name: block_size, dtype: float64
Blocks <= 100: 4410491
Blocks <= 500: 4412724
Blocks <= 1000: 4412983

address_numbers_sizes
count    915055.000000
mean         10.232141
std         203.208232
min           1.000000
50%           1.000000
75%           3.000000
90%           7.000000
95%          15.000000
99%         115.000000
max       54339.000000
Name: block_size, dtype: float64

In [1]:
# ============================================================
# AMAZON ML CHALLENGE 2026
# BUSINESS ENTITY RESOLUTION - MASTER PIPELINE
# ============================================================

import os
import re
import gc
import time
import sqlite3
import unicodedata
import numpy as np
import pandas as pd

from difflib import SequenceMatcher
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

PROJECT_PATH = r"C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026"

TRAIN_PATH = os.path.join(PROJECT_PATH, "data", "raw", "train")
TEST_PATH = os.path.join(PROJECT_PATH, "data", "raw", "test")

DB_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "blocking",
    "entity_blocking.db"
)

OUTPUT_DIR = os.path.join(
    PROJECT_PATH,
    "submission",
    "output"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

MATCHING_PATH = os.path.join(
    OUTPUT_DIR,
    "matching_results.tsv"
)

CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "candidate_pairs.tsv"
)

THRESHOLD = 0.15

START_TIME = time.time()

print("=" * 70)
print("AMAZON ML CHALLENGE 2026 - MASTER PIPELINE")
print("=" * 70)

# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKC", text)

    text = "".join(
        " " if unicodedata.category(ch).startswith(("P", "S")) else ch
        for ch in text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_address_numbers(text):
    if pd.isna(text):
        return ""
    numbers = re.findall(r"\d+", str(text))
    return "_".join(numbers[:2])


def extract_first_address_number(text):
    if pd.isna(text):
        return ""

    numbers = re.findall(r"\d+", str(text))

    if len(numbers) == 0:
        return ""

    return numbers[0]


def similarity_ratio(a, b):
    return SequenceMatcher(
        None,
        str(a),
        str(b)
    ).ratio() * 100


def token_sort_similarity(a, b):
    a_tokens = str(a).split()
    b_tokens = str(b).split()

    a_sorted = " ".join(sorted(a_tokens))
    b_sorted = " ".join(sorted(b_tokens))

    return SequenceMatcher(
        None,
        a_sorted,
        b_sorted
    ).ratio() * 100


# ------------------------------------------------------------
# 2. VERIFY EXISTING BLOCKING DATABASE
# ------------------------------------------------------------

print("\n[1/8] Checking existing blocking database...")

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"Blocking database not found:\n{DB_PATH}"
    )

print(
    "Database size:",
    round(os.path.getsize(DB_PATH) / (1024**3), 2),
    "GB"
)

conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

print("Tables:")
print(tables["name"].tolist())

# ------------------------------------------------------------
# 3. LOAD TEST SOURCE 1
# ------------------------------------------------------------

print("\n[2/8] Loading test Source 1...")

test1 = pd.read_csv(
    os.path.join(TEST_PATH, "test_source1.tsv"),
    sep="\t",
    dtype=str
)

print("Test S1:", test1.shape)

# ------------------------------------------------------------
# 4. CREATE TEST BLOCKING KEYS
# ------------------------------------------------------------

print("\n[3/8] Creating test blocking keys...")

test1["name_norm"] = (
    test1["business_name"]
    .apply(normalize_text)
)

test1["name_prefix"] = (
    test1["name_norm"]
    .str.replace(" ", "", regex=False)
    .str[:4]
)

test1["block_key"] = (
    test1["country"]
    .fillna("")
    .astype(str)
    + "_"
    + test1["name_prefix"]
    .fillna("")
)

test1["address_norm"] = (
    test1["business_address"]
    .apply(normalize_text)
)

test1["address_prefix"] = (
    test1["address_norm"]
    .str[:8]
)

test1["address_numbers"] = (
    test1["business_address"]
    .apply(extract_address_numbers)
)

test1["first_number"] = (
    test1["business_address"]
    .apply(extract_first_address_number)
)

test1["number_block"] = (
    test1["country"]
    .fillna("")
    .astype(str)
    + "_"
    + test1["first_number"]
)

print("Blocking keys ready.")

# ------------------------------------------------------------
# 5. REBUILD ALLOWED BLOCKS FROM EXISTING SQLITE SIZE TABLES
# ------------------------------------------------------------

print("\n[4/8] Loading selective block definitions...")

BLOCK_LIMIT_NAME = 500
BLOCK_LIMIT_ADDRESS_NUMBERS = 500
BLOCK_LIMIT_NUMBER = 500
BLOCK_LIMIT_ADDRESS_PREFIX = 100

allowed_blocks = {}

# Name block
allowed_blocks["block_key_sizes"] = set(
    pd.read_sql_query(
        f"""
        SELECT block_key
        FROM block_key_sizes
        WHERE size <= {BLOCK_LIMIT_NAME}
        """,
        conn
    )["block_key"]
)

# Address prefix
allowed_blocks["address_prefix_sizes"] = set(
    pd.read_sql_query(
        f"""
        SELECT address_prefix
        FROM address_prefix_sizes
        WHERE size <= {BLOCK_LIMIT_ADDRESS_PREFIX}
        """,
        conn
    )["address_prefix"]
)

# Address numbers
allowed_blocks["address_numbers_sizes"] = set(
    pd.read_sql_query(
        f"""
        SELECT address_numbers
        FROM address_numbers_sizes
        WHERE size <= {BLOCK_LIMIT_ADDRESS_NUMBERS}
        """,
        conn
    )["address_numbers"]
)

# Number block
allowed_blocks["number_block_sizes"] = set(
    pd.read_sql_query(
        f"""
        SELECT number_block
        FROM number_block_sizes
        WHERE size <= {BLOCK_LIMIT_NUMBER}
        """,
        conn
    )["number_block"]
)

# VERY IMPORTANT:
# Empty blocks are never valid blocking keys.
for key in allowed_blocks:
    allowed_blocks[key] = {
        x for x in allowed_blocks[key]
        if x not in ("", "_", "US_", "India_", "France_")
    }

print(
    "Allowed name blocks:",
    len(allowed_blocks["block_key_sizes"])
)

print(
    "Allowed address prefixes:",
    len(allowed_blocks["address_prefix_sizes"])
)

print(
    "Allowed address-number blocks:",
    len(allowed_blocks["address_numbers_sizes"])
)

print(
    "Allowed number blocks:",
    len(allowed_blocks["number_block_sizes"])
)

# ------------------------------------------------------------
# 6. BUILD CANDIDATE LOOKUP DICTIONARIES
# ------------------------------------------------------------

print("\n[5/8] Building candidate lookup dictionaries...")
print("This uses the existing SQLite database.")

def load_lookup(table_name, key_column, allowed_values, batch_size=5000):

    allowed_values = list(allowed_values)

    lookup = {}

    total = len(allowed_values)

    for start in range(0, total, batch_size):

        batch = allowed_values[start:start + batch_size]

        placeholders = ",".join(["?"] * len(batch))

        query = f"""
            SELECT {key_column}, entity_id
            FROM {table_name}
            WHERE {key_column} IN ({placeholders})
              AND source IN ('TEST2','TEST3')
        """

        rows = conn.execute(
            query,
            batch
        ).fetchall()

        for key, entity_id in rows:

            if key not in lookup:
                lookup[key] = []

            lookup[key].append(entity_id)

        if start % (batch_size * 10) == 0:
            print(
                f"  {min(start + batch_size, total):,}/{total:,}"
            )

    return lookup


name_lookup = load_lookup(
    "block_key_sizes",
    "block_key",
    allowed_blocks["block_key_sizes"]
)

print(
    "Name lookup blocks:",
    len(name_lookup)
)

address_prefix_lookup = load_lookup(
    "address_prefix_sizes",
    "address_prefix",
    allowed_blocks["address_prefix_sizes"]
)

print(
    "Address-prefix lookup blocks:",
    len(address_prefix_lookup)
)

address_numbers_lookup = load_lookup(
    "address_numbers_sizes",
    "address_numbers",
    allowed_blocks["address_numbers_sizes"]
)

print(
    "Address-number lookup blocks:",
    len(address_numbers_lookup)
)

number_lookup = load_lookup(
    "number_block_sizes",
    "number_block",
    allowed_blocks["number_block_sizes"]
)

print(
    "Number lookup blocks:",
    len(number_lookup)
)

# ------------------------------------------------------------
# 7. GENERATE FINAL CANDIDATES
# ------------------------------------------------------------

print("\n[6/8] Generating final candidate pairs...")

candidate_rows = []

candidate_counts = []

for i, row in enumerate(
    test1[
        [
            "entity_id",
            "block_key",
            "address_prefix",
            "address_numbers",
            "number_block"
        ]
    ].itertuples(index=False)
):

    s1_id = row.entity_id

    candidates = set()

    # Name block
    if row.block_key in allowed_blocks["block_key_sizes"]:
        candidates.update(
            name_lookup.get(row.block_key, [])
        )

    # Address prefix
    if row.address_prefix in allowed_blocks["address_prefix_sizes"]:
        candidates.update(
            address_prefix_lookup.get(
                row.address_prefix,
                []
            )
        )

    # Address numbers
    if row.address_numbers in allowed_blocks["address_numbers_sizes"]:
        candidates.update(
            address_numbers_lookup.get(
                row.address_numbers,
                []
            )
        )

    # First address number
    if row.number_block in allowed_blocks["number_block_sizes"]:
        candidates.update(
            number_lookup.get(
                row.number_block,
                []
            )
        )

    candidates = list(candidates)

    candidate_counts.append(len(candidates))

    for candidate_id in candidates:
        candidate_rows.append(
            (
                s1_id,
                candidate_id
            )
        )

    if (i + 1) % 100000 == 0:
        print(
            f"  Processed {i + 1:,}/{len(test1):,} S1 | "
            f"candidates generated: {len(candidate_rows):,}"
        )

candidate_df = pd.DataFrame(
    candidate_rows,
    columns=[
        "source1_entity_id",
        "candidate_entity_id"
    ]
)

print("\nCandidate generation finished.")

print(
    "Total candidate pairs:",
    f"{len(candidate_df):,}"
)

print(
    "Average candidates/S1:",
    round(
        len(candidate_df) / len(test1),
        2
    )
)

print(
    "Median candidates/S1:",
    int(np.median(candidate_counts))
)

print(
    "Max candidates/S1:",
    int(np.max(candidate_counts))
)

# ------------------------------------------------------------
# 8. RETRAIN PAIR CLASSIFIER
# ------------------------------------------------------------

print("\n[7/8] Retraining XGBoost pair classifier...")

# Load training data
train1 = pd.read_csv(
    os.path.join(TRAIN_PATH, "train_source1.tsv"),
    sep="\t",
    dtype=str
)

train2 = pd.read_csv(
    os.path.join(TRAIN_PATH, "train_source2.tsv"),
    sep="\t",
    dtype=str
)

train3 = pd.read_csv(
    os.path.join(TRAIN_PATH, "train_source3.tsv"),
    sep="\t",
    dtype=str
)

train_gr = pd.read_csv(
    os.path.join(TRAIN_PATH, "train_ground_truth.tsv"),
    sep="\t",
    dtype=str
)

print(
    "Training data loaded:",
    len(train1),
    len(train2),
    len(train3)
)

# ------------------------------------------------------------
# Build 25k positive + 25k hard-negative training pairs
# ------------------------------------------------------------

gt_mapping = {}

for row in train_gr.itertuples(index=False):

    matched = row.matched_entity_ids

    if pd.isna(matched):
        gt_mapping[row.source1_entity_id] = set()

    else:
        gt_mapping[row.source1_entity_id] = set(
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        )

POSITIVE_TARGET = 25000

positive_rows = []

for row in train_gr.itertuples(index=False):

    matched = row.matched_entity_ids

    if pd.isna(matched):
        continue

    for candidate_id in str(matched).split(","):

        candidate_id = candidate_id.strip()

        if candidate_id:

            positive_rows.append(
                (
                    row.source1_entity_id,
                    candidate_id,
                    1
                )
            )

        if len(positive_rows) >= POSITIVE_TARGET:
            break

    if len(positive_rows) >= POSITIVE_TARGET:
        break

positive_df = pd.DataFrame(
    positive_rows,
    columns=[
        "s1_id",
        "candidate_id",
        "target"
    ]
)

# ------------------------------------------------------------
# Hard negatives using existing training blocking database
# ------------------------------------------------------------

print("Creating hard negatives...")

# Build normalized S1 training lookup
train1_small = train1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

train1_small["name_norm"] = (
    train1_small["business_name"]
    .apply(normalize_text)
)

train1_small["name_prefix"] = (
    train1_small["name_norm"]
    .str.replace(" ", "", regex=False)
    .str[:4]
)

train1_small["block_key"] = (
    train1_small["country"]
    .fillna("")
    .astype(str)
    + "_"
    + train1_small["name_prefix"]
)

train1_small["address_norm"] = (
    train1_small["business_address"]
    .apply(normalize_text)
)

train1_small["address_prefix"] = (
    train1_small["address_norm"]
    .str[:8]
)

train1_small["address_numbers"] = (
    train1_small["business_address"]
    .apply(extract_address_numbers)
)

train1_small["first_number"] = (
    train1_small["business_address"]
    .apply(extract_first_address_number)
)

train1_small["number_block"] = (
    train1_small["country"]
    .fillna("")
    .astype(str)
    + "_"
    + train1_small["first_number"]
)

positive_set = set(
    zip(
        positive_df["s1_id"],
        positive_df["candidate_id"]
    )
)

negative_rows = []

rng = np.random.default_rng(42)

# Use a random sample of S1 rows for negative generation
sample_s1 = train1_small.sample(
    n=min(50000, len(train1_small)),
    random_state=42
)

for row in sample_s1.itertuples(index=False):

    if len(negative_rows) >= 25000:
        break

    candidate_ids = set()

    # Prefer selective blocks
    if row.address_prefix:
        result = conn.execute(
            """
            SELECT entity_id
            FROM entity_blocks
            WHERE address_prefix = ?
              AND source IN ('TEST2','TEST3')
            LIMIT 100
            """,
            (row.address_prefix,)
        ).fetchall()

        candidate_ids.update(
            x[0] for x in result
        )

    if row.address_numbers:
        result = conn.execute(
            """
            SELECT entity_id
            FROM entity_blocks
            WHERE address_numbers = ?
              AND source IN ('TEST2','TEST3')
            LIMIT 100
            """,
            (row.address_numbers,)
        ).fetchall()

        candidate_ids.update(
            x[0] for x in result
        )

    # Instead of relying on TEST IDs, create negatives
    # directly from train Source 2/3 below.

# ------------------------------------------------------------
# More reliable negative creation directly from train S2/S3
# ------------------------------------------------------------

candidate_pool = pd.concat(
    [
        train2[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        train3[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ]
    ],
    ignore_index=True
)

# Sample a manageable negative pool
negative_pool = candidate_pool.sample(
    n=min(100000, len(candidate_pool)),
    random_state=42
)

# Random S1 rows
negative_s1 = train1_small.sample(
    n=min(50000, len(train1_small)),
    random_state=43
)

negative_pairs = []

for s1_row in negative_s1.itertuples(index=False):

    if len(negative_pairs) >= 25000:
        break

    # Try same country first
    same_country = negative_pool[
        negative_pool["country"].fillna("").astype(str)
        == str(s1_row.country)
    ]

    if len(same_country) == 0:
        same_country = negative_pool

    candidate = same_country.sample(
        n=1,
        random_state=int(
            rng.integers(0, 2**31 - 1)
        )
    ).iloc[0]

    pair = (
        s1_row.entity_id,
        candidate.entity_id
    )

    if pair in positive_set:
        continue

    negative_pairs.append(
        (
            s1_row.entity_id,
            candidate.entity_id,
            0
        )
    )

negative_df = pd.DataFrame(
    negative_pairs,
    columns=[
        "s1_id",
        "candidate_id",
        "target"
    ]
)

print(
    "Positive pairs:",
    len(positive_df)
)

print(
    "Negative pairs:",
    len(negative_df)
)

ml_pairs = pd.concat(
    [
        positive_df,
        negative_df
    ],
    ignore_index=True
)

ml_pairs = ml_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Retrieve candidate records
# ------------------------------------------------------------

print("Preparing ML training features...")

needed_s1 = set(
    ml_pairs["s1_id"]
)

s1_features = train1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

s1_features = s1_features[
    s1_features["entity_id"].isin(needed_s1)
].copy()

needed_candidates = set(
    ml_pairs["candidate_id"]
)

candidate_features = pd.concat(
    [
        train2[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        train3[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ]
    ],
    ignore_index=True
)

candidate_features = candidate_features[
    candidate_features["entity_id"].isin(
        needed_candidates
    )
].drop_duplicates(
    subset=["entity_id"]
)

feature_df = ml_pairs.merge(
    s1_features,
    left_on="s1_id",
    right_on="entity_id",
    how="left"
).drop(
    columns=["entity_id"]
)

feature_df = feature_df.merge(
    candidate_features,
    left_on="candidate_id",
    right_on="entity_id",
    how="left",
    suffixes=("_s1", "_candidate")
).drop(
    columns=["entity_id"]
)

# ------------------------------------------------------------
# Feature engineering
# ------------------------------------------------------------

feature_df["name_s1_norm"] = (
    feature_df["business_name_s1"]
    .apply(normalize_text)
)

feature_df["name_candidate_norm"] = (
    feature_df["business_name_candidate"]
    .apply(normalize_text)
)

feature_df["address_s1_norm"] = (
    feature_df["business_address_s1"]
    .apply(normalize_text)
)

feature_df["address_candidate_norm"] = (
    feature_df["business_address_candidate"]
    .apply(normalize_text)
)

feature_df["name_ratio"] = [
    similarity_ratio(a, b)
    for a, b in zip(
        feature_df["name_s1_norm"],
        feature_df["name_candidate_norm"]
    )
]

feature_df["name_token_sort_ratio"] = [
    token_sort_similarity(a, b)
    for a, b in zip(
        feature_df["name_s1_norm"],
        feature_df["name_candidate_norm"]
    )
]

feature_df["address_ratio"] = [
    similarity_ratio(a, b)
    for a, b in zip(
        feature_df["address_s1_norm"],
        feature_df["address_candidate_norm"]
    )
]

feature_df["address_token_sort_ratio"] = [
    token_sort_similarity(a, b)
    for a, b in zip(
        feature_df["address_s1_norm"],
        feature_df["address_candidate_norm"]
    )
]

feature_df["country_match"] = (
    feature_df["country_s1"]
    .fillna("")
    .str.lower()
    ==
    feature_df["country_candidate"]
    .fillna("")
    .str.lower()
).astype(int)

feature_df["name_exact_match"] = (
    feature_df["name_s1_norm"]
    ==
    feature_df["name_candidate_norm"]
).astype(int)

feature_df["address_exact_match"] = (
    feature_df["address_s1_norm"]
    ==
    feature_df["address_candidate_norm"]
).astype(int)

feature_df["name_contains"] = feature_df.apply(
    lambda row: int(
        row["name_s1_norm"] != ""
        and
        row["name_candidate_norm"] != ""
        and
        (
            row["name_s1_norm"]
            in
            row["name_candidate_norm"]
            or
            row["name_candidate_norm"]
            in
            row["name_s1_norm"]
        )
    ),
    axis=1
)

feature_df["address_contains"] = feature_df.apply(
    lambda row: int(
        row["address_s1_norm"] != ""
        and
        row["address_candidate_norm"] != ""
        and
        (
            row["address_s1_norm"]
            in
            row["address_candidate_norm"]
            or
            row["address_candidate_norm"]
            in
            row["address_s1_norm"]
        )
    ),
    axis=1
)

feature_df["name_length_diff"] = (
    feature_df["name_s1_norm"].str.len()
    -
    feature_df["name_candidate_norm"].str.len()
).abs()

feature_df["address_length_diff"] = (
    feature_df["address_s1_norm"].str.len()
    -
    feature_df["address_candidate_norm"].str.len()
).abs()

def first_number(text):
    numbers = re.findall(
        r"\d+",
        str(text)
    )

    return numbers[0] if numbers else ""

feature_df["first_number_s1"] = (
    feature_df["address_s1_norm"]
    .apply(first_number)
)

feature_df["first_number_candidate"] = (
    feature_df["address_candidate_norm"]
    .apply(first_number)
)

feature_df["first_number_match"] = (
    (
        feature_df["first_number_s1"] != ""
    )
    &
    (
        feature_df["first_number_s1"]
        ==
        feature_df["first_number_candidate"]
    )
).astype(int)

feature_columns = [
    "name_ratio",
    "name_token_sort_ratio",
    "address_ratio",
    "address_token_sort_ratio",
    "country_match",
    "name_exact_match",
    "address_exact_match",
    "name_contains",
    "address_contains",
    "name_length_diff",
    "address_length_diff",
    "first_number_match"
]

X = feature_df[
    feature_columns
].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y = feature_df["target"]

# ------------------------------------------------------------
# Group split
# ------------------------------------------------------------

groups = feature_df["s1_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

# ------------------------------------------------------------
# Train model
# ------------------------------------------------------------

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_val, y_val)
    ],
    verbose=False
)

val_proba = model.predict_proba(
    X_val
)[:, 1]

print(
    "Validation ROC-AUC:",
    round(
        roc_auc_score(
            y_val,
            val_proba
        ),
        4
    )
)

# ------------------------------------------------------------
# 9. TEST INFERENCE
# ------------------------------------------------------------

print("\n[8/8] Running test inference...")

# Build candidate feature data from test S2/S3
print("Loading test Source 2 and Source 3...")

test2 = pd.read_csv(
    os.path.join(
        TEST_PATH,
        "test_source2.tsv"
    ),
    sep="\t",
    dtype=str
)

test3 = pd.read_csv(
    os.path.join(
        TEST_PATH,
        "test_source3.tsv"
    ),
    sep="\t",
    dtype=str
)

test_candidates = pd.concat(
    [
        test2,
        test3
    ],
    ignore_index=True
)

test_candidates = test_candidates[
    test_candidates["entity_id"].isin(
        set(candidate_df["candidate_entity_id"])
    )
].drop_duplicates(
    subset=["entity_id"]
)

print(
    "Unique candidate records loaded:",
    len(test_candidates)
)

# ------------------------------------------------------------
# Process candidates in chunks
# ------------------------------------------------------------

candidate_id_set = set(
    candidate_df["candidate_entity_id"]
)

test_s1_lookup = test1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

test_s1_lookup = test_s1_lookup.rename(
    columns={
        "entity_id": "s1_id",
        "business_name": "business_name_s1",
        "business_address": "business_address_s1",
        "country": "country_s1"
    }
)

test_candidates = test_candidates.rename(
    columns={
        "entity_id": "candidate_id",
        "business_name": "business_name_candidate",
        "business_address": "business_address_candidate",
        "country": "country_candidate"
    }
)

# Map candidate records
candidate_lookup = test_candidates.set_index(
    "candidate_id"
)[
    [
        "business_name_candidate",
        "business_address_candidate",
        "country_candidate"
    ]
].to_dict(
    "index"
)

# We process S1 groups in chunks so memory stays controlled.
MATCH_ROWS = []

candidate_df = candidate_df.drop_duplicates()

CHUNK_SIZE = 10000

for start in range(
    0,
    len(test1),
    CHUNK_SIZE
):

    s1_chunk = test1.iloc[
        start:start + CHUNK_SIZE
    ][
        [
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ].copy()

    s1_chunk = s1_chunk.rename(
        columns={
            "entity_id": "s1_id",
            "business_name": "business_name_s1",
            "business_address": "business_address_s1",
            "country": "country_s1"
        }
    )

    chunk_ids = set(
        s1_chunk["s1_id"]
    )

    pairs = candidate_df[
        candidate_df["source1_entity_id"].isin(
            chunk_ids
        )
    ].copy()

    if len(pairs) == 0:
        continue

    pairs = pairs.merge(
        s1_chunk,
        left_on="source1_entity_id",
        right_on="s1_id",
        how="left"
    )

    candidate_info = []

    valid_pair_indexes = []

    for idx, candidate_id in enumerate(
        pairs["candidate_entity_id"]
    ):

        info = candidate_lookup.get(
            candidate_id
        )

        if info is not None:

            candidate_info.append(info)
            valid_pair_indexes.append(idx)

    if not valid_pair_indexes:
        continue

    pairs = pairs.iloc[
        valid_pair_indexes
    ].copy()

    candidate_info_df = pd.DataFrame(
        candidate_info,
        index=pairs.index
    )

    pairs[
        [
            "business_name_candidate",
            "business_address_candidate",
            "country_candidate"
        ]
    ] = candidate_info_df[
        [
            "business_name_candidate",
            "business_address_candidate",
            "country_candidate"
        ]
    ]

    # Normalize
    pairs["name_s1_norm"] = (
        pairs["business_name_s1"]
        .apply(normalize_text)
    )

    pairs["name_candidate_norm"] = (
        pairs["business_name_candidate"]
        .apply(normalize_text)
    )

    pairs["address_s1_norm"] = (
        pairs["business_address_s1"]
        .apply(normalize_text)
    )

    pairs["address_candidate_norm"] = (
        pairs["business_address_candidate"]
        .apply(normalize_text)
    )

    pairs["name_ratio"] = [
        similarity_ratio(a, b)
        for a, b in zip(
            pairs["name_s1_norm"],
            pairs["name_candidate_norm"]
        )
    ]

    pairs["name_token_sort_ratio"] = [
        token_sort_similarity(a, b)
        for a, b in zip(
            pairs["name_s1_norm"],
            pairs["name_candidate_norm"]
        )
    ]

    pairs["address_ratio"] = [
        similarity_ratio(a, b)
        for a, b in zip(
            pairs["address_s1_norm"],
            pairs["address_candidate_norm"]
        )
    ]

    pairs["address_token_sort_ratio"] = [
        token_sort_similarity(a, b)
        for a, b in zip(
            pairs["address_s1_norm"],
            pairs["address_candidate_norm"]
        )
    ]

    pairs["country_match"] = (
        pairs["country_s1"]
        .fillna("")
        .str.lower()
        ==
        pairs["country_candidate"]
        .fillna("")
        .str.lower()
    ).astype(int)

    pairs["name_exact_match"] = (
        pairs["name_s1_norm"]
        ==
        pairs["name_candidate_norm"]
    ).astype(int)

    pairs["address_exact_match"] = (
        pairs["address_s1_norm"]
        ==
        pairs["address_candidate_norm"]
    ).astype(int)

    pairs["name_contains"] = pairs.apply(
        lambda row: int(
            row["name_s1_norm"] != ""
            and
            row["name_candidate_norm"] != ""
            and
            (
                row["name_s1_norm"]
                in
                row["name_candidate_norm"]
                or
                row["name_candidate_norm"]
                in
                row["name_s1_norm"]
            )
        ),
        axis=1
    )

    pairs["address_contains"] = pairs.apply(
        lambda row: int(
            row["address_s1_norm"] != ""
            and
            row["address_candidate_norm"] != ""
            and
            (
                row["address_s1_norm"]
                in
                row["address_candidate_norm"]
                or
                row["address_candidate_norm"]
                in
                row["address_s1_norm"]
            )
        ),
        axis=1
    )

    pairs["name_length_diff"] = (
        pairs["name_s1_norm"].str.len()
        -
        pairs["name_candidate_norm"].str.len()
    ).abs()

    pairs["address_length_diff"] = (
        pairs["address_s1_norm"].str.len()
        -
        pairs["address_candidate_norm"].str.len()
    ).abs()

    pairs["first_number_s1"] = (
        pairs["address_s1_norm"]
        .apply(first_number)
    )

    pairs["first_number_candidate"] = (
        pairs["address_candidate_norm"]
        .apply(first_number)
    )

    pairs["first_number_match"] = (
        (
            pairs["first_number_s1"] != ""
        )
        &
        (
            pairs["first_number_s1"]
            ==
            pairs["first_number_candidate"]
        )
    ).astype(int)

    X_test_chunk = pairs[
        feature_columns
    ].replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    probabilities = model.predict_proba(
        X_test_chunk
    )[:, 1]

    pairs["probability"] = probabilities

    matches = pairs[
        pairs["probability"] >= THRESHOLD
    ][
        [
            "source1_entity_id",
            "candidate_entity_id",
            "probability"
        ]
    ]

    MATCH_ROWS.extend(
        matches.itertuples(
            index=False,
            name=None
        )
    )

    if (
        start // CHUNK_SIZE
    ) % 10 == 0:

        print(
            f"  Inference: "
            f"{min(start + CHUNK_SIZE, len(test1)):,}/"
            f"{len(test1):,} S1 | "
            f"matches so far: {len(MATCH_ROWS):,}"
        )

    del pairs
    del X_test_chunk

    gc.collect()

# ------------------------------------------------------------
# 10. CREATE MATCHING RESULTS
# ------------------------------------------------------------

print("\nCreating matching_results.tsv...")

match_df = pd.DataFrame(
    MATCH_ROWS,
    columns=[
        "source1_entity_id",
        "candidate_entity_id",
        "probability"
    ]
)

if len(match_df) > 0:

    match_df = match_df.sort_values(
        [
            "source1_entity_id",
            "probability"
        ],
        ascending=[
            True,
            False
        ]
    )

    # Remove duplicate candidate IDs
    match_df = match_df.drop_duplicates(
        subset=[
            "source1_entity_id",
            "candidate_entity_id"
        ]
    )

    grouped_matches = (
        match_df
        .groupby(
            "source1_entity_id"
        )["candidate_entity_id"]
        .apply(
            lambda x: ",".join(
                dict.fromkeys(x.tolist())
            )
        )
    )

else:

    grouped_matches = pd.Series(
        dtype=str
    )

matching_results = pd.DataFrame(
    {
        "source1_entity_id":
            test1["entity_id"]
    }
)

matching_results["matched_entity_ids"] = (
    matching_results["source1_entity_id"]
    .map(grouped_matches)
    .fillna("")
)

matching_results.to_csv(
    MATCHING_PATH,
    sep="\t",
    index=False
)

# ------------------------------------------------------------
# 11. CREATE CANDIDATE PAIRS
# ------------------------------------------------------------

print("Creating candidate_pairs.tsv...")

candidate_grouped = (
    candidate_df
    .groupby(
        "source1_entity_id"
    )["candidate_entity_id"]
    .apply(
        lambda x: ",".join(
            dict.fromkeys(x.tolist())
        )
    )
)

candidate_results = pd.DataFrame(
    {
        "source1_entity_id":
            test1["entity_id"]
    }
)

candidate_results["candidate_entity_ids"] = (
    candidate_results["source1_entity_id"]
    .map(candidate_grouped)
    .fillna("")
)

candidate_results.to_csv(
    CANDIDATE_PATH,
    sep="\t",
    index=False
)

# ------------------------------------------------------------
# 12. FINAL SANITY CHECKS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SANITY CHECKS")
print("=" * 70)

print(
    "Test S1 rows:",
    f"{len(test1):,}"
)

print(
    "Matching result rows:",
    f"{len(matching_results):,}"
)

print(
    "Candidate result rows:",
    f"{len(candidate_results):,}"
)

print(
    "Matching rows correct:",
    len(matching_results) == len(test1)
)

print(
    "Candidate rows correct:",
    len(candidate_results) == len(test1)
)

# Check duplicates in final matches
duplicate_match_pairs = 0

if len(match_df) > 0:

    duplicate_match_pairs = (
        match_df
        .duplicated(
            subset=[
                "source1_entity_id",
                "candidate_entity_id"
            ]
        )
        .sum()
    )

print(
    "Duplicate match pairs:",
    duplicate_match_pairs
)

# Check every predicted match exists in candidates
if len(match_df) > 0:

    candidate_pair_set = set(
        zip(
            candidate_df["source1_entity_id"],
            candidate_df["candidate_entity_id"]
        )
    )

    predicted_pair_set = set(
        zip(
            match_df["source1_entity_id"],
            match_df["candidate_entity_id"]
        )
    )

    outside_candidates = (
        predicted_pair_set
        -
        candidate_pair_set
    )

else:

    outside_candidates = set()

print(
    "Predictions outside candidate set:",
    len(outside_candidates)
)

# Check empty predictions
empty_predictions = (
    matching_results["matched_entity_ids"]
    == ""
).sum()

print(
    "S1 entities with no predicted match:",
    f"{empty_predictions:,}"
)

# ------------------------------------------------------------
# 13. FILE SIZES
# ------------------------------------------------------------

print("\nOutput files:")

for path in [
    MATCHING_PATH,
    CANDIDATE_PATH
]:

    print(
        os.path.basename(path),
        ":",
        round(
            os.path.getsize(path) / (1024**2),
            2
        ),
        "MB"
    )

elapsed = (
    time.time()
    -
    START_TIME
)

print(
    "\nTotal runtime:",
    round(elapsed / 60, 2),
    "minutes"
)

print("\nDONE.")
print("Matching:", MATCHING_PATH)
print("Candidates:", CANDIDATE_PATH)

AMAZON ML CHALLENGE 2026 - MASTER PIPELINE

[1/8] Checking existing blocking database...
Database size: 10.85 GB
Tables:
['entity_blocks', 'block_key_sizes', 'address_prefix_sizes', 'address_numbers_sizes', 'number_block_sizes', 'candidates_10000', 'temp_test_s1']

[2/8] Loading test Source 1...
Test S1: (1732544, 4)

[3/8] Creating test blocking keys...
Blocking keys ready.

[4/8] Loading selective block definitions...


DatabaseError: Execution failed on sql '
        SELECT block_key
        FROM block_key_sizes
        WHERE size <= 500
        ': no such column: block_key

In [2]:
for table in [
    "block_key_sizes",
    "address_prefix_sizes",
    "address_numbers_sizes",
    "number_block_sizes",
    "temp_test_s1",
    "candidates_10000"
]:
    print(f"\n--- {table} ---")
    print(pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    ))


--- block_key_sizes ---
   cid         name  type  notnull dflt_value  pk
0    0  block_value  TEXT        0       None   0
1    1   block_size              0       None   0

--- address_prefix_sizes ---
   cid         name  type  notnull dflt_value  pk
0    0  block_value  TEXT        0       None   0
1    1   block_size              0       None   0

--- address_numbers_sizes ---
   cid         name  type  notnull dflt_value  pk
0    0  block_value  TEXT        0       None   0
1    1   block_size              0       None   0

--- number_block_sizes ---
   cid         name  type  notnull dflt_value  pk
0    0  block_value  TEXT        0       None   0
1    1   block_size              0       None   0

--- temp_test_s1 ---
   cid             name  type  notnull dflt_value  pk
0    0        entity_id  TEXT        0       None   0
1    1        block_key  TEXT        0       None   0
2    2   address_prefix  TEXT        0       None   0
3    3  address_numbers  TEXT        0       Non

In [ ]:
print("Candidate rows:", conn.execute(
    "SELECT COUNT(*) FROM candidates_10000"
).fetchone()[0])

print("Unique S1:", conn.execute(
    "SELECT COUNT(DISTINCT s1_id) FROM candidates_10000"
).fetchone()[0])

print("Unique candidates:", conn.execute(
    "SELECT COUNT(DISTINCT candidate_id) FROM candidates_10000"
).fetchone()[0])

print("\nS1 range:")
print(conn.execute(
    "SELECT MIN(s1_id), MAX(s1_id) FROM candidates_10000"
).fetchone())

print("\nFirst 5 rows:")
print(pd.read_sql_query(
    """
    SELECT *
    FROM candidates_10000
    LIMIT 5
    """,
    conn
))

Candidate rows: 240044071
Unique S1: 10000
